> The monitoring data and building-specific configuration are confidential and are not included. See `README.md` for execution requirements. Clear any generated outputs before sharing an executed notebook.


In [ ]:
# Private workspace and configuration
from copy import deepcopy as _privacy_deepcopy
from functools import lru_cache as _privacy_lru_cache
import json
import os
from pathlib import Path


def _private_config_path(variable):
    value = os.environ.get(variable)
    if not value:
        raise RuntimeError(f"Set {variable} before running this notebook; see README.md.")
    return Path(value).expanduser().resolve()


def private_workspace():
    """Return the configured private input/output directory."""
    path = _private_config_path("HVAC_PRIVATE_WORKSPACE")
    if not path.is_dir():
        raise FileNotFoundError("The configured private workspace does not exist.")
    return path


def _decode_private_values(value):
    if isinstance(value, list):
        return [_decode_private_values(item) for item in value]
    if isinstance(value, dict):
        if set(value) == {"__tuple_key_dict__"}:
            return {tuple(key): _decode_private_values(item)
                    for key, item in value["__tuple_key_dict__"]}
        return {key: _decode_private_values(item) for key, item in value.items()}
    return value


@_privacy_lru_cache(maxsize=1)
def _load_private_values():
    with _private_config_path("HVAC_PRIVATE_SETTINGS").open(encoding="utf-8") as handle:
        values = json.load(handle).get("private_values")
    if not isinstance(values, dict):
        raise ValueError("Private settings must contain a private_values object.")
    return _decode_private_values(values)


def private_value(key):
    """Read a private setting without embedding its value in the public source."""
    values = _load_private_values()
    if key not in values or values[key] is None:
        raise KeyError(f"Required private setting is missing: {key!r}.")
    return _privacy_deepcopy(values[key])


# Notebook 07: Neural training stability across random seeds

This notebook assesses variation in the Shared NN caused by parameter initialization, dropout and minibatch order. It retains the data, chronological split, features, target scales, architecture, loss, optimization settings and duration-evaluation procedure used in Notebooks 01–03.

The seed set is \(\{42,142,242,342,442\}\). Seed 42 is loaded from the original checkpoint; the remaining four models are trained independently. Every outcome is reported, and test errors do not select the reference seed or a replacement model.

The configured experiment retains the original batch sizes and hardware checks. Each replicate selects its checkpoint using the same validation-duration criterion. Saved reference predictions are verified before the across-seed comparisons are accepted.

The summaries cover aggregate duration accuracy, event-averaged errors, prediction-level dispersion and secondary temperature/rate accuracy. Across-seed variation measures training stability under this fixed procedure; it is not calibrated predictive uncertainty or external validation. In the code, `phase` denotes operating mode and an anchor identifies a duration prediction time.


In [ ]:


# Imports, reference hardware/training settings, paths

import os, re, json, math, random, warnings, shutil, hashlib, time, sys, platform
from pathlib import Path
from itertools import cycle
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
import sklearn
import joblib

try:
    from IPython.display import display
except Exception:
    def display(x):
        print(x)

try:
    import torch
    import torch.nn as nn
    from torch.utils.data import Dataset
except Exception as e:
    raise RuntimeError('PyTorch is required for Notebook 07.') from e

try:
    from google.colab import drive  # type: ignore
    if not Path('/content/drive/MyDrive').exists():
        drive.mount('/content/drive')
except Exception:
    pass

def safe_torch_load(path, map_location=None):
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=map_location)

def sha256_file(path, chunk_size=1024 * 1024):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        while True:
            b = f.read(chunk_size)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

warnings.filterwarnings('default')
warnings.filterwarnings(
    'ignore',
    message=r'datetime\.datetime\.utcnow\(\) is deprecated.*',
    category=DeprecationWarning,
)

# Seed set is fixed before TEST evaluation.

SEEDS = [42, 142, 242, 342, 442]
CANONICAL_SEED = 42
NEW_SEEDS = [s for s in SEEDS if s != CANONICAL_SEED]

if len(SEEDS) != len(set(SEEDS)) or CANONICAL_SEED not in SEEDS:
    raise RuntimeError('Invalid pre-specified seed set.')

STRICT_DETERMINISM = False  # Reference setting.

def reset_seeds(seed):
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

# Reference device behavior.

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

if DEVICE.type == 'cuda':
    torch.set_float32_matmul_precision('high')
    if hasattr(torch.backends, 'cuda'):
        torch.backends.cuda.matmul.allow_tf32 = True
    props = torch.cuda.get_device_properties(0)
    GPU_NAME = props.name
    GPU_MEMORY_GB = props.total_memory / 1024**3
else:
    GPU_NAME = 'CPU'
    GPU_MEMORY_GB = 0.0

if STRICT_DETERMINISM:
    torch.use_deterministic_algorithms(True, warn_only=True)
    if torch.backends.cudnn.is_available():
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
else:
    if torch.backends.cudnn.is_available():
        torch.backends.cudnn.benchmark = True

USE_AMP = bool(DEVICE.type == 'cuda')
AMP_DTYPE = torch.float16

# Retain the reference GPU and batch configuration for the additional training seeds.
CANONICAL_GPU_REFERENCE = 'Tesla T4'
CANONICAL_BATCH_TRAJECTORIES_PER_PHASE = 256
CANONICAL_BATCH_DURATION_ANCHORS_PER_PHASE = 256

# Keep this strict by default so the new seed runs differ from the reference
# experiment by random seed rather than by effective batch size or GPU family.
REQUIRE_T4_FOR_NEW_SEEDS = True

if NEW_SEEDS:
    if DEVICE.type != 'cuda':
        raise RuntimeError(
            'New PAPER-FINAL seed replicates require a CUDA GPU. '
            'The canonical run used a Tesla T4.'
        )
    if GPU_MEMORY_GB < 10:
        raise RuntimeError(
            f'GPU memory {GPU_MEMORY_GB:.1f} GB is below the canonical '
            'batch-size requirement. Do not silently reduce batch size.'
        )
    if REQUIRE_T4_FOR_NEW_SEEDS and 'T4' not in GPU_NAME.upper():
        raise RuntimeError(
            f'Current GPU is {GPU_NAME!r}; canonical seed 42 was trained on '
            'a Tesla T4. For a seed-only stability experiment, reconnect to '
            'a T4 runtime. If a different GPU must be used, set '
            'REQUIRE_T4_FOR_NEW_SEEDS=False and report the hardware change.'
        )

# Private reference workspace.

BASE_DIR = Path(str(private_workspace()))
if 'PAPER-TEMP-NEW' in str(BASE_DIR):
    raise RuntimeError('STOP: Notebook 07 must use PAPER-TEMP, not PAPER-TEMP-NEW.')

DATASET_FOLDER = 'full_rampup_5min_feature_dataset_UNIFIED_DERIVATIVE_DURATION_AWARE'
DATA_DIR = BASE_DIR / DATASET_FOLDER
SPLIT_MAP_PATH = DATA_DIR / 'shared_event_split_map.csv'
SPLIT_SIGNATURE_PATH = DATA_DIR / 'shared_event_split_signature.json'

CANONICAL_TRAIN_RUN = (
    BASE_DIR /
    'training_UNIFIED_DERIVATIVE_DURATION_AWARE_PAPER_FINAL' /
    'paper_final_q_weighted'
)
CANONICAL_MODEL_DIR = CANONICAL_TRAIN_RUN / 'models'
CANONICAL_CHECKPOINT = CANONICAL_MODEL_DIR / 'unified_directed_rate_nn.pt'
CANONICAL_TRAIN_MANIFEST = (
    CANONICAL_TRAIN_RUN /
    'training_manifest_UNIFIED_DERIVATIVE_DURATION_AWARE_PAPER_FINAL.json'
)

NOTEBOOK03_RUN_ID = private_value('shared.notebook03_run_id')
NOTEBOOK03_RUN = (
    BASE_DIR /
    'duration_UNIFIED_DERIVATIVE_DURATION_AWARE_PAPER_FINAL' /
    NOTEBOOK03_RUN_ID
)
NB03_ALL_PRED_PATH = (
    NOTEBOOK03_RUN /
    'predictions' /
    'ALL_remaining_duration_predictions_with_relative_errors.csv'
)
NB03_MAIN_PATH = (
    NOTEBOOK03_RUN /
    'tables' /
    'PAPER_MAIN_duration_metrics_EXACT_COMMON_SUPPORT.csv'
)
NB03_MANIFEST_PATH = (
    NOTEBOOK03_RUN /
    'duration_manifest_UNIFIED_DERIVATIVE_DURATION_AWARE_PAPER_FINAL.json'
)

# Notebook 07 never overwrites the reference paper run.
OUT_ROOT = (
    BASE_DIR /
    'multiseed_UNIFIED_DERIVATIVE_DURATION_AWARE_PAPER_FINAL' /
    'paper_final_5seed_stability'
)
COMMON_DIR = OUT_ROOT / 'common'
MODEL_DIR = COMMON_DIR / 'models'  # copied Notebook-02 preprocessing writes here
TABLE_DIR = OUT_ROOT / 'tables'
FIG_DIR = OUT_ROOT / 'figures'
PRED_DIR = OUT_ROOT / 'predictions'
AUDIT_DIR = OUT_ROOT / 'audit'
SEED_ROOT = OUT_ROOT / 'seeds'

for p in [
    COMMON_DIR, MODEL_DIR, TABLE_DIR, FIG_DIR,
    PRED_DIR, AUDIT_DIR, SEED_ROOT
]:
    p.mkdir(parents=True, exist_ok=True)

# Reference scientific/training constants.

PHASES = ['heating', 'cooling']
STEP_MIN = 5.0
END_BAND_C = 0.4
DURATION_MAX_HORIZON_BY_PHASE = {
    'heating': 185.0,
    'cooling': 65.0,
}

NN_HIDDEN_DIMS = [128, 64, 32]
NN_DROPOUT = 0.10
NN_LR = 1e-3
NN_WEIGHT_DECAY = 1e-4

# Fixed to the ACTUAL reference seed-42 run, not dynamically changed by hardware.
NN_BATCH_TRAJECTORIES_PER_PHASE = CANONICAL_BATCH_TRAJECTORIES_PER_PHASE
NN_BATCH_DURATION_ANCHORS_PER_PHASE = CANONICAL_BATCH_DURATION_ANCHORS_PER_PHASE

NN_MAX_EPOCHS = 500
NN_VALIDATION_EVERY = 2
NN_EARLY_STOP_PATIENCE_VALIDATIONS = 10
NN_SCHEDULER_PATIENCE_VALIDATIONS = 2
NN_SCHEDULER_FACTOR = 0.5
NN_MIN_LR = 1e-6
NN_MIN_DELTA = 1e-5
SELECTED_SCORE_MODE = 'score_weighted'

SOFT_TEMP_BETA_C = 0.04
SOFT_DIRECTION_BETA_FLOOR_C = 0.005

# Exact Notebook-03 hard-converter constants.
INDIRECT_MAX_HORIZON_BY_PHASE = {
    'heating': 185.0,
    'cooling': 65.0,
}
EXTRAPOLATE_FINAL_N_POINTS = 4
EXTRAPOLATE_MIN_ABS_SLOPE_C_PER_MIN = 1e-4
EXTRAPOLATE_MAX_EXTRA_MIN_BY_PHASE = {
    'heating': 60.0,
    'cooling': 30.0,
}

# Resumability.
RESUME_TRAINING = True
FORCE_CONTINUE_AFTER_EARLY_STOP = False
RESET_SEEDS = []  # e.g. [242] only if deliberately restarting that seed.

# Strict replay guards.
STRICT_CANONICAL_REPLAY = True
CANONICAL_REPLAY_TOL_MIN = 1e-3

print('DEVICE:', DEVICE)
print('GPU:', GPU_NAME, f'({GPU_MEMORY_GB:.1f} GB)')
print('DATA_DIR:', DATA_DIR)
print('Canonical training run:', CANONICAL_TRAIN_RUN)
print('Notebook 03 run:', NOTEBOOK03_RUN)
print('Notebook 07 output:', OUT_ROOT)
print('Pre-specified seeds:', SEEDS)
print(
    'Fixed canonical batches:',
    NN_BATCH_TRAJECTORIES_PER_PHASE,
    NN_BATCH_DURATION_ANCHORS_PER_PHASE
)


## 1. Load the reference datasets and split

Use Notebook 02's data-loading procedure for temperature/rate trajectories and dedicated duration anchors. Verify that all rows inherit the saved event-level chronological assignment.


In [ ]:

# Load long-format temperature trajectories + saved duration anchors + shared split

def norm_event_id(x):
    if pd.isna(x): raise RuntimeError('Missing event_id')
    s = str(x).strip()
    if re.fullmatch(r'[-+]?\d+\.0+', s): s = str(int(float(s)))
    return s

def load_phase_long(phase):
    phase_dir = DATA_DIR/phase
    main = phase_dir/f'{phase}_full_rampup_5min_samples_LONG.csv'
    by_room = sorted((phase_dir/'by_room').glob(f'*_{phase}_full_rampup_5min_LONG.csv'))
    frames=[]
    if main.exists(): frames.append(pd.read_csv(main))
    frames += [pd.read_csv(p) for p in by_room]
    if not frames: raise FileNotFoundError(f'No LONG data found for {phase}')
    d=pd.concat(frames,ignore_index=True,sort=False)
    d['phase']=phase
    for c in ['timestamp','target_timestamp','rampup_start_time','rampup_end_time','rampup_end_grid_time','extraction_end_time']:
        if c in d.columns: d[c]=pd.to_datetime(d[c],errors='coerce')
    d['event_id']=d['event_id'].map(norm_event_id)
    d['room']=d['room'].astype(str).str.strip()
    d['shared_event_uid']=d['phase']+'|'+d['room']+'|'+d['event_id']
    key=['phase','room','event_id','timestamp','target_timestamp','horizon_min']
    return d.drop_duplicates(subset=[c for c in key if c in d.columns],keep='first').reset_index(drop=True)

split_map=pd.read_csv(SPLIT_MAP_PATH)
split_map['event_id']=split_map['event_id'].map(norm_event_id)
split_map['room']=split_map['room'].astype(str).str.strip()
split_map['shared_event_uid']=split_map['phase'].astype(str)+'|'+split_map['room']+'|'+split_map['event_id']
split_map=split_map[['shared_event_uid','split','split_block']].drop_duplicates()

# long-format temperature/rate supervision.
phase_frames=[]
for phase in PHASES:
    d=load_phase_long(phase)
    d=d.merge(split_map,on='shared_event_uid',how='left',validate='many_to_one')
    if d['split'].isna().any():
        bad=d[d['split'].isna()].copy(); bad.to_csv(AUDIT_DIR/f'{phase}_BAD_unmatched_LONG_split.csv',index=False)
        raise RuntimeError(f'{phase}: LONG rows do not match authoritative split map.')
    d['event_key']=d['shared_event_uid']
    d['target_context']=(d['target_context'] if 'target_context' in d.columns else pd.Series('inside_event',index=d.index)).astype(str).str.strip().str.lower()
    phase_frames.append(d)

DATA=pd.concat(phase_frames,ignore_index=True,sort=False)
DATA['_row_id']=np.arange(len(DATA),dtype=int)
DATA['use_for_temperature_training']=DATA['target_context'].isin(['inside_event','post_end_trend_continuation','post_end_buffer'])

required_long=['phase','room','event_key','split','timestamp','target_timestamp','horizon_min','Room_T','target_Room_T','thermal_mode_sign','Phase_Target_T_C','Directed_Distance_to_Target_C','Room_dTdt_now_bwd2_C_per_min','target_directed_rate_C_per_min']
missing=[c for c in required_long if c not in DATA.columns]
if missing: raise RuntimeError(f'Notebook 01 LONG columns missing: {missing}')
for c in ['horizon_min','Room_T','target_Room_T','thermal_mode_sign','Phase_Target_T_C','Directed_Distance_to_Target_C','Room_dTdt_now_bwd2_C_per_min','target_directed_rate_C_per_min']:
    DATA[c]=pd.to_numeric(DATA[c],errors='coerce')

# Dedicated duration rows define the duration supervision set independently of
# temperature-target rows.
anchor_frames=[]
for phase in PHASES:
    p=DATA_DIR/phase/f'{phase}_remaining_duration_anchors_5min.csv'
    if not p.exists(): raise FileNotFoundError(p)
    a=pd.read_csv(p); a['phase']=phase
    for c in ['timestamp','anchor_timestamp','feature_source_timestamp','rampup_start_time','rampup_end_time','rampup_end_grid_time']:
        if c in a.columns: a[c]=pd.to_datetime(a[c],errors='coerce')
    a['event_id']=a['event_id'].map(norm_event_id)
    a['room']=a['room'].astype(str).str.strip()
    a['shared_event_uid']=a['phase']+'|'+a['room']+'|'+a['event_id']
    a=a.merge(split_map,on='shared_event_uid',how='left',validate='many_to_one')
    if a['split'].isna().any():
        bad=a[a['split'].isna()].copy(); bad.to_csv(AUDIT_DIR/f'{phase}_BAD_unmatched_duration_split.csv',index=False)
        raise RuntimeError(f'{phase}: duration anchors do not match authoritative split map.')
    a['event_key']=a['shared_event_uid']
    a['true_remaining_duration_min']=pd.to_numeric(a.get('true_remaining_duration_min',a.get('remaining_duration_min')),errors='coerce')
    a['remaining_trajectory_quality_score']=pd.to_numeric(a['remaining_trajectory_quality_score'],errors='coerce').clip(0.0,1.0)
    a['thermal_mode_sign']=pd.to_numeric(a['thermal_mode_sign'],errors='coerce')
    a['Phase_Target_T_C']=pd.to_numeric(a['Phase_Target_T_C'],errors='coerce')
    a['Room_T']=pd.to_numeric(a['Room_T'],errors='coerce')
    a['asof_gap_min']=(a['anchor_timestamp']-a['feature_source_timestamp']).dt.total_seconds()/60.0
    a['asof_gap_min']=a['asof_gap_min'].clip(lower=0.0)
    kind=(a['anchor_kind'] if 'anchor_kind' in a.columns else pd.Series('anchor',index=a.index)).astype(str)
    a['anchor_kind']=kind
    a['anchor_key']=a['event_key']+'|'+a['anchor_timestamp'].astype(str)+'|'+kind
    a['Duration_Crossing_Threshold_C']=a['Phase_Target_T_C']-a['thermal_mode_sign']*END_BAND_C
    anchor_frames.append(a)

ANCHORS=pd.concat(anchor_frames,ignore_index=True,sort=False)
ANCHORS=ANCHORS.drop_duplicates('anchor_key',keep='first').reset_index(drop=True)
required_anchor=['phase','room','event_key','split','anchor_key','anchor_timestamp','feature_source_timestamp','Room_T','Phase_Target_T_C','thermal_mode_sign','true_remaining_duration_min','remaining_trajectory_quality_score','asof_gap_min']
missing=[c for c in required_anchor if c not in ANCHORS.columns]
if missing: raise RuntimeError(f'Notebook 01 duration-anchor columns missing: {missing}')

print('LONG rows:',len(DATA),'events:',DATA['event_key'].nunique(),'rooms:',DATA['room'].nunique())
display(DATA.groupby(['phase','split']).agg(rows=('_row_id','size'),events=('event_key','nunique'),rooms=('room','nunique')))
print('\nAUTHORITATIVE duration anchors:',len(ANCHORS),'events:',ANCHORS['event_key'].nunique())
display(ANCHORS.groupby(['phase','split']).agg(anchors=('anchor_key','nunique'),events=('event_key','nunique'),rooms=('room','nunique')))

# Audit that event-start anchors are present rather than silently lost by long-format
# horizon requirements.
if 'anchor_kind' in ANCHORS.columns:
    display(ANCHORS.groupby(['phase','split','anchor_kind']).size().rename('anchors').reset_index())

# Reference Consistency check — every model uses this exact split

def canonical_split_signature(split_frame):
    z=split_frame.copy()
    if 'shared_event_uid' not in z.columns:
        z['shared_event_uid']=z['phase'].astype(str)+'|'+z['room'].astype(str).str.strip()+'|'+z['event_id'].astype(str).str.strip()
    # Re-expand event UID only for a stable fingerprint compatible with Notebook 01.
    parts=z['shared_event_uid'].astype(str).str.split('|',n=2,expand=True)
    sig=pd.DataFrame({
        'phase':parts[0],
        'room':parts[1],
        'event_id':parts[2],
        'split':z['split'].astype(str),
        'split_block':z['split_block'].astype(str),
    }).drop_duplicates().sort_values(['phase','room','event_id']).reset_index(drop=True)
    canonical=sig.to_csv(index=False,lineterminator='\n')
    return hashlib.sha256(canonical.encode('utf-8')).hexdigest(),sig

SHARED_SPLIT_SHA256,SHARED_SPLIT_CANONICAL=canonical_split_signature(split_map)
if SPLIT_SIGNATURE_PATH.exists():
    with open(SPLIT_SIGNATURE_PATH) as f:
        expected_sig=json.load(f)
    if SHARED_SPLIT_SHA256!=expected_sig.get('sha256'):
        raise RuntimeError('Shared split SHA256 differs from Notebook 01. STOP: models would not use the same chronological split.')

# Verify that temperature and duration rows inherit the saved event split.
for label,frame in [('LONG',DATA),('DURATION_ANCHORS',ANCHORS)]:
    if frame['split'].isna().any(): raise RuntimeError(f'{label}: missing shared split assignments.')
    if not set(frame['split'].unique()).issubset({'train','valid','test'}): raise RuntimeError(f'{label}: unexpected split labels.')

split_audit=(
    SHARED_SPLIT_CANONICAL.groupby(['phase','split'],as_index=False)
    .size().rename(columns={'size':'events'})
)
split_audit.to_csv(AUDIT_DIR/'AUTHORITATIVE_shared_split_counts.csv',index=False)
with open(AUDIT_DIR/'AUTHORITATIVE_shared_split_signature.json','w') as f:
    json.dump({'sha256':SHARED_SPLIT_SHA256,'counts':split_audit.to_dict(orient='records')},f,indent=2)
print('Verified authoritative shared split SHA256:',SHARED_SPLIT_SHA256)
display(split_audit)


## 2. Verify input features

Apply the same input-selection rules and feature order to every seed. Features are not selected separately for individual replicates.


In [ ]:

# Select inputs without explicit room identifiers or raw temperature-history lags

DERIVATIVE_HISTORY_LAGS_MINUTES = list(range(5,65,5))
SOURCE_DERIVATIVE_HISTORY_LAGS_MINUTES = list(range(5,30,5))

COMMON_INPUTS = [
    'horizon_min',
    'Room_T', 'Room_V', 'Phase_Target_T_C', 'Directed_Distance_to_Target_C',
    'T_ext', 'T_ave_other_rooms',
    'Thermal_Source_T_C', 'Directed_Source_Room_DeltaT_C', 'Thermal_Input_Proxy',
    'thermal_mode_sign',
    'Room_dTdt_now_bwd2_C_per_min', 'Room_d2Tdt2_now_bwd2_C_per_min2',
    'hour_decimal','day_of_year','day_of_week','month','is_weekend',
    'doy_sin','doy_cos','dow_sin','dow_cos','month_sin','month_cos',
    'solar_declination_proxy_deg','daylight_hours_proxy','sunrise_hour_proxy','sunset_hour_proxy',
    'daylight_fraction_proxy','time_since_sunrise_h_proxy','time_to_sunset_h_proxy','solar_elevation_proxy',
]
COMMON_INPUTS += [f'Room_dTdt_central_-{lag}min_C_per_min' for lag in DERIVATIVE_HISTORY_LAGS_MINUTES]
COMMON_INPUTS += [f'Room_d2Tdt2_central_-{lag}min_C_per_min2' for lag in DERIVATIVE_HISTORY_LAGS_MINUTES]
COMMON_INPUTS += [f'ThermalSource_dTdt_central_-{lag}min_C_per_min' for lag in SOURCE_DERIVATIVE_HISTORY_LAGS_MINUTES]
COMMON_INPUTS += [f'ThermalSource_d2Tdt2_central_-{lag}min_C_per_min2' for lag in SOURCE_DERIVATIVE_HISTORY_LAGS_MINUTES]

# Explicitly forbidden future/label/identity variables.
FORBIDDEN_EXACT = {
    'room','room_label','event_id','event_key','shared_event_uid','target_Room_T','target_timestamp',
    'target_directed_rate_C_per_min','target_Room_dTdt_central_C_per_min',
    'remaining_time_to_end_min','remaining_duration_min','true_remaining_duration_min',
    'remaining_trajectory_quality_score','remaining_forward_progress_C','remaining_reverse_motion_C',
    'remaining_total_variation_C','remaining_quality_n_steps','rampup_start_time','rampup_end_time',
    'rampup_end_grid_time','extraction_end_time','split','split_block',
}
forbidden_pattern = re.compile(r'^Room_T -\d+min$|^Room_T \+\d+min$|^room__', re.I)

train_all=DATA[(DATA['split']=='train') & DATA['use_for_temperature_training']].copy()
anchor_train_all=ANCHORS[ANCHORS['split'].eq('train')].copy()
selected=[]; audit=[]
for c in COMMON_INPUTS:
    if c in FORBIDDEN_EXACT or forbidden_pattern.search(c):
        audit.append({'feature':c,'status':'forbidden'}); continue
    if c not in DATA.columns:
        audit.append({'feature':c,'status':'missing_LONG'}); continue
    if c!='horizon_min' and c not in ANCHORS.columns:
        audit.append({'feature':c,'status':'missing_duration_anchor'}); continue
    tr=pd.to_numeric(train_all[c],errors='coerce').replace([np.inf,-np.inf],np.nan)
    if tr.notna().sum()==0:
        audit.append({'feature':c,'status':'all_missing_LONG_train'}); continue
    if c!='horizon_min':
        atr=pd.to_numeric(anchor_train_all[c],errors='coerce').replace([np.inf,-np.inf],np.nan)
        if atr.notna().sum()==0:
            audit.append({'feature':c,'status':'all_missing_duration_anchor_train'}); continue
    phase_counts={p:int(pd.to_numeric(train_all.loc[train_all['phase'].eq(p),c],errors='coerce').replace([np.inf,-np.inf],np.nan).notna().sum()) for p in PHASES}
    if any(v==0 for v in phase_counts.values()):
        audit.append({'feature':c,'status':'missing_in_one_phase_train',**{f'finite_{p}':v for p,v in phase_counts.items()}}); continue
    if tr.dropna().nunique()<=1 and c!='thermal_mode_sign':
        audit.append({'feature':c,'status':'zero_variance_train'}); continue
    selected.append(c); audit.append({'feature':c,'status':'selected','missing_fraction_train':float(tr.isna().mean()),**{f'finite_{p}':v for p,v in phase_counts.items()}})

# thermal_mode_sign must vary in the combined training set; this confirms one network
# sees both modes.
if 'thermal_mode_sign' not in selected: selected.append('thermal_mode_sign')
if any(forbidden_pattern.search(c) for c in selected): raise RuntimeError('Raw room-temperature lag or room identity slipped into model inputs.')
if any(c in FORBIDDEN_EXACT for c in selected): raise RuntimeError('Forbidden target/identity input slipped into model inputs.')

pd.DataFrame(audit).to_csv(AUDIT_DIR/'unified_model_input_audit.csv',index=False)
pd.DataFrame({'feature':selected}).to_csv(AUDIT_DIR/'unified_model_inputs_ordered.csv',index=False)
MODEL_INPUTS=selected
print('Selected shared numeric inputs:',len(MODEL_INPUTS))
print(MODEL_INPUTS)


## 3. Verify preprocessing and target scales

Recompute training-only transformations in this notebook's private output directory, then compare them with the saved seed-42 artifacts. All replicates use the same feature transformations and target normalization.


In [ ]:

# Fit training-only preprocessing and mode-specific target scales

# Mode-aware imputation is fitted from training trajectory rows only.
PHASE_MEDIANS={}
for phase in PHASES:
    z=train_all.loc[train_all['phase'].eq(phase),MODEL_INPUTS].apply(pd.to_numeric,errors='coerce').replace([np.inf,-np.inf],np.nan)
    PHASE_MEDIANS[phase]={c:float(z[c].median()) for c in MODEL_INPUTS}
    bad=[c for c,v in PHASE_MEDIANS[phase].items() if not np.isfinite(v)]
    if bad: raise RuntimeError(f'{phase}: selected inputs still have no finite TRAIN median: {bad}')

def fill_phase_medians(df):
    x=df[MODEL_INPUTS].apply(pd.to_numeric,errors='coerce').replace([np.inf,-np.inf],np.nan).copy()
    if 'phase' not in df.columns: raise RuntimeError('phase is required for phase-aware imputation.')
    phase_values=df['phase'].astype(str).to_numpy()
    for phase in PHASES:
        idx=np.where(phase_values==phase)[0]
        if len(idx)==0: continue
        for c in MODEL_INPUTS:
            col=x.columns.get_loc(c)
            vals=x.iloc[idx,col]
            x.iloc[idx,col]=vals.fillna(PHASE_MEDIANS[phase][c]).to_numpy()
    if x.isna().any().any(): raise RuntimeError('Phase-aware imputation left NaN values.')
    return x.to_numpy(dtype=float)

Xfilled=fill_phase_medians(train_all)
SCALER=StandardScaler().fit(Xfilled)
PREPROCESSOR={'numeric_cols':MODEL_INPUTS,'phase_medians':PHASE_MEDIANS,'scaler':SCALER,'include_room_identity':False,'imputation':'phase_specific_LONG_train_median'}
joblib.dump(PREPROCESSOR,MODEL_DIR/'unified_numeric_preprocessor.joblib')

def transform_X(df):
    return SCALER.transform(fill_phase_medians(df)).astype(np.float32)

def safe_std(x,floor):
    v=pd.to_numeric(pd.Series(x),errors='coerce').dropna().to_numpy(float)
    if len(v)<2: return float(floor)
    return float(max(float(np.std(v,ddof=0)),floor))

# Fit rate and temperature scales on training trajectory targets and duration scales on
# unique training prediction rows.
LOSS_SCALES={}
SOFT_DIRECTION_BETA_BY_PHASE={}
for phase in PHASES:
    tr=train_all[train_all['phase'].eq(phase)].copy()
    ar=ANCHORS[(ANCHORS['phase']==phase)&(ANCHORS['split']=='train')].copy()
    rate_scale=safe_std(tr['target_directed_rate_C_per_min'],1e-4)
    temp_scale=safe_std(tr['target_Room_T'],0.05)
    duration_scale=safe_std(ar['true_remaining_duration_min'],1.0)
    med_step=float(np.nanmedian(np.abs(pd.to_numeric(tr['target_directed_rate_C_per_min'],errors='coerce').to_numpy(float)*STEP_MIN)))
    if not np.isfinite(med_step): med_step=0.05
    beta_dir=max(SOFT_DIRECTION_BETA_FLOOR_C,0.10*med_step)
    LOSS_SCALES[phase]={'rate_C_per_min':rate_scale,'temperature_C':temp_scale,'remaining_min':duration_scale}
    SOFT_DIRECTION_BETA_BY_PHASE[phase]=float(beta_dir)

with open(MODEL_DIR/'loss_scales_train_only_by_phase.json','w') as f: json.dump(LOSS_SCALES,f,indent=2)
with open(MODEL_DIR/'soft_crossing_parameters.json','w') as f: json.dump({'temp_beta_C':SOFT_TEMP_BETA_C,'direction_beta_C_by_phase':SOFT_DIRECTION_BETA_BY_PHASE},f,indent=2)
print('TRAIN-only loss scales:'); display(pd.DataFrame(LOSS_SCALES).T)
print('Soft direction beta:',SOFT_DIRECTION_BETA_BY_PHASE)

# Score distribution is shown before any model is trained.
score_summary=(ANCHORS.groupby(['phase','split'])['remaining_trajectory_quality_score']
               .agg(['count','mean','median','std','min','max']).reset_index())
score_summary.to_csv(TABLE_DIR/'duration_quality_score_distribution.csv',index=False)
display(score_summary.round(4))


In [ ]:


# Verify against saved reference preprocessing

for p in [
    CANONICAL_CHECKPOINT,
    CANONICAL_TRAIN_MANIFEST,
    CANONICAL_MODEL_DIR / 'unified_numeric_preprocessor.joblib',
    CANONICAL_MODEL_DIR / 'loss_scales_train_only_by_phase.json',
    CANONICAL_MODEL_DIR / 'soft_crossing_parameters.json',
    NB03_ALL_PRED_PATH,
    NB03_MAIN_PATH,
    NB03_MANIFEST_PATH,
]:
    if not p.exists():
        raise FileNotFoundError(p)

CANONICAL_PREP = joblib.load(
    CANONICAL_MODEL_DIR / 'unified_numeric_preprocessor.joblib'
)

if list(CANONICAL_PREP['numeric_cols']) != list(MODEL_INPUTS):
    raise RuntimeError(
        'Notebook 07 feature order differs from frozen PAPER-FINAL seed 42.'
    )

for phase in PHASES:
    for c in MODEL_INPUTS:
        a = float(CANONICAL_PREP['phase_medians'][phase][c])
        b = float(PHASE_MEDIANS[phase][c])
        if not np.isclose(a, b, rtol=0.0, atol=1e-12):
            raise RuntimeError(
                f'Phase-median mismatch: {phase}/{c}: canonical={a}, rebuilt={b}'
            )

for attr in ['mean_', 'scale_', 'var_']:
    if not np.allclose(
        getattr(CANONICAL_PREP['scaler'], attr),
        getattr(SCALER, attr),
        rtol=0.0,
        atol=1e-12,
        equal_nan=True
    ):
        raise RuntimeError(
            f'StandardScaler {attr} differs from frozen PAPER-FINAL.'
        )

with open(
    CANONICAL_MODEL_DIR / 'loss_scales_train_only_by_phase.json',
    'r',
    encoding='utf-8'
) as f:
    CANONICAL_LOSS_SCALES = json.load(f)

with open(
    CANONICAL_MODEL_DIR / 'soft_crossing_parameters.json',
    'r',
    encoding='utf-8'
) as f:
    CANONICAL_SOFT = json.load(f)

for phase in PHASES:
    for k in ['rate_C_per_min', 'temperature_C', 'remaining_min']:
        if not np.isclose(
            float(LOSS_SCALES[phase][k]),
            float(CANONICAL_LOSS_SCALES[phase][k]),
            rtol=0.0,
            atol=1e-12
        ):
            raise RuntimeError(
                f'Loss-scale mismatch: {phase}/{k}'
            )

    if not np.isclose(
        float(SOFT_DIRECTION_BETA_BY_PHASE[phase]),
        float(CANONICAL_SOFT['direction_beta_C_by_phase'][phase]),
        rtol=0.0,
        atol=1e-12
    ):
        raise RuntimeError(
            f'Soft direction beta mismatch: {phase}'
        )

if not np.isclose(
    float(SOFT_TEMP_BETA_C),
    float(CANONICAL_SOFT['temp_beta_C']),
    rtol=0.0,
    atol=1e-12
):
    raise RuntimeError('Soft temperature beta mismatch.')

with open(CANONICAL_TRAIN_MANIFEST, 'r', encoding='utf-8') as f:
    TRAIN_MANIFEST = json.load(f)

with open(NB03_MANIFEST_PATH, 'r', encoding='utf-8') as f:
    NB03_MANIFEST = json.load(f)

if TRAIN_MANIFEST.get('shared_split_sha256') != SHARED_SPLIT_SHA256:
    raise RuntimeError('Frozen Notebook-02 split differs from Notebook 07.')
if NB03_MANIFEST.get('shared_split_sha256') != SHARED_SPLIT_SHA256:
    raise RuntimeError('Frozen Notebook-03 split differs from Notebook 07.')

CANONICAL_CKPT = safe_torch_load(
    CANONICAL_CHECKPOINT,
    map_location='cpu'
)

if CANONICAL_CKPT.get('shared_split_sha256') != SHARED_SPLIT_SHA256:
    raise RuntimeError('Canonical seed-42 checkpoint split mismatch.')

if int(CANONICAL_CKPT['n_inputs']) != len(MODEL_INPUTS):
    raise RuntimeError('Canonical seed-42 input dimension mismatch.')

if list(CANONICAL_CKPT['hidden_dims']) != NN_HIDDEN_DIMS:
    raise RuntimeError('Canonical hidden architecture mismatch.')

if not np.isclose(
    float(CANONICAL_CKPT['dropout']),
    NN_DROPOUT,
    atol=0,
    rtol=0
):
    raise RuntimeError('Canonical dropout mismatch.')

print('Frozen PAPER-FINAL preprocessing/loss/checkpoint verification: PASSED')
print('Canonical seed-42 best epoch:', CANONICAL_CKPT.get('best_epoch'))


## 4. Build trajectory and duration supervision

Construct the two supervision datasets using Notebook 02's procedure. Check duration-row coverage independently from the availability of future temperature targets.


In [ ]:

# Build two supervision streams for the shared NN
#    A) long-format trajectories -> directed-rate + reconstructed-T loss
#    B) saved duration anchors -> remaining-time loss

DATA_X=transform_X(DATA)

class TemperatureTrajectoryDataset(Dataset):
    def __init__(self,frame,Xall):
        self.items=[]
        d=frame[frame['use_for_temperature_training']].copy()
        for (phase,event_key,timestamp),g in d.groupby(['phase','event_key','timestamp'],sort=False):
            g=g.sort_values('horizon_min').drop_duplicates('horizon_min',keep='first')
            h=pd.to_numeric(g['horizon_min'],errors='coerce').to_numpy(float)
            valid=np.isfinite(h)&(h>0)
            g=g.iloc[np.where(valid)[0]].copy(); h=h[valid]
            if len(g)==0: continue
            order=np.argsort(h); g=g.iloc[order].copy(); h=h[order]
            if np.any(np.abs(h/STEP_MIN-np.round(h/STEP_MIN))>1e-8): continue
            rid=g['_row_id'].to_numpy(int)
            target_rate=pd.to_numeric(g['target_directed_rate_C_per_min'],errors='coerce').to_numpy(float)
            target_T=pd.to_numeric(g['target_Room_T'],errors='coerce').to_numpy(float)
            rate_mask=np.isfinite(target_rate); temp_mask=np.isfinite(target_T)
            if not temp_mask.any() or not rate_mask.any(): continue
            first=g.iloc[0]
            v0_phys=pd.to_numeric(first.get('Room_dTdt_now_bwd2_C_per_min',np.nan),errors='coerce')
            if not np.isfinite(v0_phys):
                past=pd.to_numeric(first.get('Room_dTdt_central_-5min_C_per_min',np.nan),errors='coerce')
                v0_phys=float(past) if np.isfinite(past) else 0.0
            self.items.append({
                'phase':str(phase),'event_key':str(event_key),'timestamp':pd.Timestamp(timestamp),
                'row_ids':rid,'X':Xall[rid],'h':h.astype(np.float32),
                'target_rate':target_rate.astype(np.float32),'rate_mask':rate_mask,
                'target_T':target_T.astype(np.float32),'temp_mask':temp_mask,
                'T0':float(first['Room_T']),'sign':float(first['thermal_mode_sign']),'v0_phys':float(v0_phys),
            })
    def __len__(self): return len(self.items)
    def __getitem__(self,i): return self.items[i]

def horizon_grid(phase):
    return np.arange(STEP_MIN,DURATION_MAX_HORIZON_BY_PHASE[phase]+1e-9,STEP_MIN,dtype=float)

def repeat_anchor_for_horizons(row,phase):
    h=horizon_grid(phase)
    syn=pd.DataFrame([row.to_dict()]*len(h))
    syn['horizon_min']=h
    syn['target_timestamp']=pd.Timestamp(row['feature_source_timestamp'])+pd.to_timedelta(h,unit='m')
    return h,syn

class DurationAnchorDataset(Dataset):
    def __init__(self,frame):
        self.items=[]
        for _,row in frame.iterrows():
            phase=str(row['phase'])
            true_rem=pd.to_numeric(row['true_remaining_duration_min'],errors='coerce')
            T0=pd.to_numeric(row['Room_T'],errors='coerce')
            sign=pd.to_numeric(row['thermal_mode_sign'],errors='coerce')
            target=pd.to_numeric(row['Phase_Target_T_C'],errors='coerce')
            gap=pd.to_numeric(row['asof_gap_min'],errors='coerce')
            q=pd.to_numeric(row['remaining_trajectory_quality_score'],errors='coerce')
            if not all(np.isfinite(v) for v in [true_rem,T0,sign,target,gap,q]): continue
            h,syn=repeat_anchor_for_horizons(row,phase)
            try:
                X=transform_X(syn)
            except Exception as e:
                raise RuntimeError(f"Failed to build synthetic inputs for anchor {row.get('anchor_key','?')}: {e}") from e
            v0=pd.to_numeric(row.get('Room_dTdt_now_bwd2_C_per_min',np.nan),errors='coerce')
            if not np.isfinite(v0):
                v0=pd.to_numeric(row.get('Room_dTdt_central_-5min_C_per_min',0.0),errors='coerce')
                if not np.isfinite(v0): v0=0.0
            self.items.append({
                'phase':phase,'anchor_key':str(row['anchor_key']),'event_key':str(row['event_key']),
                'anchor_kind':str(row.get('anchor_kind','anchor')),'X':X,'h':h.astype(np.float32),
                'T0':float(T0),'sign':float(sign),'v0_phys':float(v0),'phase_target':float(target),
                'true_remaining':float(true_rem),'asof_gap':float(gap),'q':float(np.clip(q,0,1)),
            })
    def __len__(self): return len(self.items)
    def __getitem__(self,i): return self.items[i]

TEMP_DATASETS={}
DURATION_DATASETS={}
for split in ['train','valid','test']:
    for phase in PHASES:
        tds=TemperatureTrajectoryDataset(DATA[(DATA['split']==split)&(DATA['phase']==phase)],DATA_X)
        TEMP_DATASETS[(split,phase)]=tds
        ads=DurationAnchorDataset(ANCHORS[(ANCHORS['split']==split)&(ANCHORS['phase']==phase)])
        DURATION_DATASETS[(split,phase)]=ads
        print(split,phase,'temperature issue trajectories:',len(tds),'| authoritative duration anchors:',len(ads))
        if split in ['train','valid'] and (len(tds)==0 or len(ads)==0):
            raise RuntimeError(f'Empty required supervision dataset for {split}/{phase}')

# Verify that duration supervision covers every required prediction row.
anchor_audit=[]
for split in ['train','valid','test']:
    for phase in PHASES:
        expected=int(ANCHORS[(ANCHORS['split']==split)&(ANCHORS['phase']==phase)]['anchor_key'].nunique())
        actual=len(DURATION_DATASETS[(split,phase)])
        anchor_audit.append({'phase':phase,'split':split,'authoritative_anchor_count':expected,'duration_dataset_count':actual,'difference':actual-expected})
anchor_audit=pd.DataFrame(anchor_audit)
anchor_audit.to_csv(AUDIT_DIR/'authoritative_duration_supervision_coverage.csv',index=False)
display(anchor_audit)
if (anchor_audit['difference']!=0).any():
    raise RuntimeError('Some authoritative duration anchors were lost while building duration supervision; inspect audit before training.')


## 5. Reuse the network and loss

Retain the reference architecture, directed-rate integration, smooth training completion-time calculation, regularity-weighted duration loss and validation metrics.


In [ ]:

# Unified directed-rate NN + fully vectorized GPU losses

class UnifiedDirectedRateNN(nn.Module):
    def __init__(self,n_in,hidden=(128,64,32),dropout=0.10):
        super().__init__(); layers=[]; prev=n_in
        for h in hidden:
            layers += [nn.Linear(prev,h),nn.Tanh()]
            if dropout>0: layers.append(nn.Dropout(dropout))
            prev=h
        layers.append(nn.Linear(prev,1)); self.net=nn.Sequential(*layers)
    def forward(self,x): return self.net(x).squeeze(-1)

def _model_forward_fp32(model,x):
    # Use mixed precision for network operations and 32-bit floats for integration and
    # losses.
    with torch.autocast(device_type=DEVICE.type,dtype=AMP_DTYPE,enabled=USE_AMP):
        y=model(x)
    return y.float()

def make_batches(ds,batch_size,shuffle=True,seed=0):
    idx=np.arange(len(ds))
    if shuffle: np.random.default_rng(seed).shuffle(idx)
    return [[ds.items[j] for j in idx[i:i+batch_size]] for i in range(0,len(idx),batch_size)]

def stack_trajectory_batch(batch):
    B=len(batch); H=max(len(x['h']) for x in batch); F=batch[0]['X'].shape[1]
    X=np.zeros((B,H,F),np.float32); h=np.zeros((B,H),np.float32)
    target_rate=np.zeros((B,H),np.float32); target_T=np.zeros((B,H),np.float32)
    valid=np.zeros((B,H),bool); rmask=np.zeros((B,H),bool); tmask=np.zeros((B,H),bool)
    row_ids=[]
    T0=np.zeros(B,np.float32); sign=np.zeros(B,np.float32); v0=np.zeros(B,np.float32)
    for b,item in enumerate(batch):
        n=len(item['h']); X[b,:n]=item['X']; h[b,:n]=item['h']; valid[b,:n]=True
        if n<H: h[b,n:]=item['h'][-1]  # Repeated padded endpoints contribute zero integration time.
        target_rate[b,:n]=item['target_rate']; target_T[b,:n]=item['target_T']
        rmask[b,:n]=item['rate_mask']; tmask[b,:n]=item['temp_mask']
        T0[b]=item['T0']; sign[b]=item['sign']; v0[b]=item['v0_phys']; row_ids.append(item['row_ids'])
    def tt(a,dtype=torch.float32): return torch.as_tensor(a,dtype=dtype,device=DEVICE)
    return {'X':tt(X),'h':tt(h),'target_rate':tt(target_rate),'target_T':tt(target_T),
            'valid':tt(valid,torch.bool),'rmask':tt(rmask,torch.bool),'tmask':tt(tmask,torch.bool),
            'T0':tt(T0),'sign':tt(sign),'v0':tt(v0),'row_ids':row_ids}

def integrate_variable_batch_torch(h,pred_directed,T0,sign,v0):
    pred_phys=sign[:,None]*pred_directed.float()
    dt=torch.cat([h[:,:1],h[:,1:]-h[:,:-1]],dim=1).float()
    prev_rate=torch.cat([v0[:,None],pred_phys[:,:-1]],dim=1)
    increments=0.5*dt*(prev_rate+pred_phys)
    return T0[:,None]+torch.cumsum(increments,dim=1)

def trajectory_batch_forward(model,batch):
    z=stack_trajectory_batch(batch); B,H,F=z['X'].shape
    pred=_model_forward_fp32(model,z['X'].reshape(B*H,F)).reshape(B,H)
    pred_T=integrate_variable_batch_torch(z['h'],pred,z['T0'],z['sign'],z['v0'])
    return z,pred,pred_T

def trajectory_batch_loss(model,batch,phase):
    z,pred,pred_T=trajectory_batch_forward(model,batch)
    rs=float(LOSS_SCALES[phase]['rate_C_per_min']); ts=float(LOSS_SCALES[phase]['temperature_C'])
    rm=z['rmask'] & z['valid']; tm=z['tmask'] & z['valid']
    if not rm.any() or not tm.any(): raise RuntimeError(f'{phase}: empty trajectory supervision mask.')
    Lr=torch.mean(((pred[rm]-z['target_rate'][rm])/rs)**2)
    Lt=torch.mean(((pred_T[tm]-z['target_T'][tm])/ts)**2)
    return {'trajectory':0.5*(Lr+Lt),'rate':Lr,'temperature':Lt}

def stack_duration_batch(batch):
    X=torch.as_tensor(np.stack([x['X'] for x in batch]).astype(np.float32),device=DEVICE)
    h=torch.as_tensor(batch[0]['h'],dtype=torch.float32,device=DEVICE)
    if any(not np.array_equal(x['h'],batch[0]['h']) for x in batch): raise RuntimeError('Duration batch horizon grids differ.')
    def tv(k): return torch.as_tensor([x[k] for x in batch],dtype=torch.float32,device=DEVICE)
    return {'X':X,'h':h,'T0':tv('T0'),'sign':tv('sign'),'v0':tv('v0_phys'),'target':tv('phase_target'),
            'true':tv('true_remaining'),'gap':tv('asof_gap'),'q':tv('q')}

def integrate_fixed_batch_torch(h,pred_directed,T0,sign,v0):
    pred_phys=sign[:,None]*pred_directed.float()
    dt=torch.cat([h[:1],h[1:]-h[:-1]]).float()
    prev_rate=torch.cat([v0[:,None],pred_phys[:,:-1]],dim=1)
    return T0[:,None]+torch.cumsum(0.5*dt[None,:]*(prev_rate+pred_phys),dim=1)

def soft_crossing_batch_torch(h,pred_T,T0,target,sign,v0,phase):
    threshold=target-sign*END_BAND_C
    temp_beta=float(SOFT_TEMP_BETA_C); dir_beta=float(SOFT_DIRECTION_BETA_BY_PHASE[phase])
    a0=torch.sigmoid(sign*(T0-threshold)/temp_beta)*torch.sigmoid(sign*v0*STEP_MIN/dir_beta)
    prev_T=torch.cat([T0[:,None],pred_T[:,:-1]],dim=1)
    a=torch.sigmoid(sign[:,None]*(pred_T-threshold[:,None])/temp_beta)*torch.sigmoid(sign[:,None]*(pred_T-prev_T)/dir_beta)
    one_minus=torch.clamp(1.0-a,min=1e-7,max=1.0)
    prefix=torch.cumprod(one_minus,dim=1)
    survival_before=torch.cat([(1-a0)[:,None],(1-a0)[:,None]*prefix[:,:-1]],dim=1)
    p_first=survival_before*a
    survival_after=(1-a0)*prefix[:,-1]
    return torch.sum(p_first*h[None,:],dim=1)+survival_after*h[-1]

def duration_batch_forward(model,batch,phase):
    z=stack_duration_batch(batch); B,H,F=z['X'].shape
    pred=_model_forward_fp32(model,z['X'].reshape(B*H,F)).reshape(B,H)
    pred_T=integrate_fixed_batch_torch(z['h'],pred,z['T0'],z['sign'],z['v0'])
    pred_source=soft_crossing_batch_torch(z['h'],pred_T,z['T0'],z['target'],z['sign'],z['v0'],phase)
    pred_anchor=torch.clamp(pred_source-z['gap'],min=0.0)
    return z,pred_anchor,pred_T,pred

def duration_batch_loss(model,batch,phase):
    z,pred,_,_=duration_batch_forward(model,batch,phase)
    ds=float(LOSS_SCALES[phase]['remaining_min'])
    sq=((pred-z['true'])/ds)**2
    w=z['q']  # Weight the training duration loss by trajectory regularity; assess its contribution in Notebook 05.
    return torch.sum(w*sq)/(torch.sum(w)+1e-8)

def evaluate_trajectory(model,ds,phase,batch_size=512):
    model.eval(); sums={'trajectory':0.0,'rate':0.0,'temperature':0.0}; weight=0
    with torch.inference_mode():
        for b in make_batches(ds,batch_size,False):
            z=trajectory_batch_loss(model,b,phase); n=len(b); weight+=n
            for k in sums: sums[k]+=float(z[k].detach().cpu())*n
    return {k:v/max(weight,1) for k,v in sums.items()}

def predict_duration_soft_dataset(model,ds,phase,batch_size=1024):
    model.eval(); rows=[]
    with torch.inference_mode():
        for batch in make_batches(ds,batch_size,False):
            z,pred,_,_=duration_batch_forward(model,batch,phase); pv=pred.cpu().numpy()
            for item,p in zip(batch,pv):
                rows.append({'anchor_key':item['anchor_key'],'event_key':item['event_key'],'anchor_kind':item['anchor_kind'],
                             'phase':phase,'true_remaining_duration_min':item['true_remaining'],
                             'pred_remaining_duration_min':float(p),'q':item['q']})
    return pd.DataFrame(rows)

def duration_eval_metrics(model,ds,phase):
    d=predict_duration_soft_dataset(model,ds,phase)
    e=d['pred_remaining_duration_min'].to_numpy(float)-d['true_remaining_duration_min'].to_numpy(float)
    denom=np.maximum(d['true_remaining_duration_min'].to_numpy(float),STEP_MIN)
    dscl=float(LOSS_SCALES[phase]['remaining_min'])
    return {
        'duration_norm_mse_unweighted':float(np.mean((e/dscl)**2)),
        'duration_MAE_min_unweighted':float(np.mean(np.abs(e))),
        'duration_RMSE_min_unweighted':float(np.sqrt(np.mean(e**2))),
        'duration_Bias_min_unweighted':float(np.mean(e)),
        'duration_Median_RAE_pct_unweighted':float(100*np.median(np.abs(e)/denom)),
    },d

# Fast implementation forward/backward check: finite forward/backward on both modes.
_test_model=UnifiedDirectedRateNN(len(MODEL_INPUTS),NN_HIDDEN_DIMS,0.0).to(DEVICE)
for _phase in PHASES:
    _b=DURATION_DATASETS[('train',_phase)].items[:min(8,len(DURATION_DATASETS[('train',_phase)].items))]
    _loss=duration_batch_loss(_test_model,_b,_phase)
    if not torch.isfinite(_loss): raise RuntimeError(f'{_phase}: non-finite vectorized duration loss.')
print('GPU-vectorized trajectory/duration utilities ready.')


## 6. Train independent seed replicates

Initialize each new model with its specified seed and use the corresponding seeded trajectory and duration-batch shuffles. Retain the remaining optimization and selection settings. The original seed-42 checkpoint is loaded rather than retrained.


In [ ]:


# Exact reference training loop parameterized by seed

def seed_paths(seed):
    root = SEED_ROOT / f'seed_{int(seed)}'
    model_dir = root / 'models'
    table_dir = root / 'tables'
    checkpoint_dir = root / 'checkpoints'
    audit_dir = root / 'audit'

    for p in [model_dir, table_dir, checkpoint_dir, audit_dir]:
        p.mkdir(parents=True, exist_ok=True)

    return {
        'root': root,
        'model_dir': model_dir,
        'table_dir': table_dir,
        'checkpoint_dir': checkpoint_dir,
        'audit_dir': audit_dir,
        'best': model_dir / 'unified_directed_rate_nn.pt',
        'resume': checkpoint_dir / 'resume_state.pt',
        'history': table_dir / 'unified_nn_training_history_score_weighted.csv',
        'complete': checkpoint_dir / 'TRAINING_COMPLETE.json',
    }

def capture_rng_state():
    st = {
        'numpy': np.random.get_state(),
        'python': random.getstate(),
        'torch': torch.get_rng_state(),
    }
    if torch.cuda.is_available():
        st['cuda'] = torch.cuda.get_rng_state_all()
    return st

def restore_rng_state(st):
    if not st:
        return

    np.random.set_state(st['numpy'])
    random.setstate(st['python'])
    torch.set_rng_state(st['torch'])

    if torch.cuda.is_available() and 'cuda' in st:
        torch.cuda.set_rng_state_all(st['cuda'])

def save_best_seed_model(
    path,
    model,
    seed,
    epoch,
    val_monitor,
    extra
):
    torch.save({
        'state_dict': {
            k: v.detach().cpu()
            for k, v in model.state_dict().items()
        },
        'n_inputs': len(MODEL_INPUTS),
        'hidden_dims': NN_HIDDEN_DIMS,
        'dropout': NN_DROPOUT,
        'score_mode': 'score_weighted',
        'random_seed': int(seed),
        'best_epoch': int(epoch),
        'best_validation_duration_norm_mse': float(val_monitor),
        'shared_split_sha256': SHARED_SPLIT_SHA256,
        'batch_trajectories_per_phase':
            NN_BATCH_TRAJECTORIES_PER_PHASE,
        'batch_duration_anchors_per_phase':
            NN_BATCH_DURATION_ANCHORS_PER_PHASE,
        'extra': extra,
    }, path)

def train_seed_model(seed):
    seed = int(seed)

    if seed == CANONICAL_SEED:
        raise RuntimeError(
            'Seed 42 is the frozen manuscript checkpoint and must not be retrained.'
        )

    paths = seed_paths(seed)

    if seed in RESET_SEEDS and paths['root'].exists():
        shutil.rmtree(paths['root'])
        paths = seed_paths(seed)
        print('Reset seed run:', seed)

    reset_seeds(seed)

    model = UnifiedDirectedRateNN(
        len(MODEL_INPUTS),
        NN_HIDDEN_DIMS,
        NN_DROPOUT
    ).to(DEVICE)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=NN_LR,
        weight_decay=NN_WEIGHT_DECAY
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode='min',
        factor=NN_SCHEDULER_FACTOR,
        patience=NN_SCHEDULER_PATIENCE_VALIDATIONS,
        min_lr=NN_MIN_LR
    )

    scaler = torch.cuda.amp.GradScaler(
        enabled=USE_AMP
    )

    start_epoch = 1
    best_val = np.inf
    best_epoch = 0
    stale_validations = 0
    history = []

    if RESUME_TRAINING and paths['resume'].exists():
        ck = safe_torch_load(
            paths['resume'],
            map_location=DEVICE
        )

        if ck.get('shared_split_sha256') != SHARED_SPLIT_SHA256:
            raise RuntimeError(
                f'Seed {seed}: resume checkpoint used a different split.'
            )

        if int(ck.get('random_seed', -1)) != seed:
            raise RuntimeError(
                f'Seed {seed}: wrong resume checkpoint random seed.'
            )

        model.load_state_dict(
            ck['model_state_dict']
        )
        optimizer.load_state_dict(
            ck['optimizer_state_dict']
        )
        scheduler.load_state_dict(
            ck['scheduler_state_dict']
        )

        if ck.get('scaler_state_dict') and USE_AMP:
            scaler.load_state_dict(
                ck['scaler_state_dict']
            )

        start_epoch = int(ck['epoch']) + 1
        best_val = float(ck['best_val'])
        best_epoch = int(ck['best_epoch'])
        stale_validations = int(
            ck['stale_validations']
        )
        history = ck.get('history', [])
        restore_rng_state(
            ck.get('rng_state')
        )

        print(
            f'Seed {seed}: resuming from epoch {start_epoch} | '
            f'best validation loss={best_val:.6f} at epoch {best_epoch}'
        )

    if (
        paths['complete'].exists() and
        paths['best'].exists() and
        not FORCE_CONTINUE_AFTER_EARLY_STOP
    ):
        print(
            f'Seed {seed}: training marked complete; loading existing best model.'
        )
        ck = safe_torch_load(
            paths['best'],
            map_location=DEVICE
        )
        model.load_state_dict(
            ck['state_dict']
        )

        if paths['history'].exists():
            history_df = pd.read_csv(paths['history'])
        else:
            history_df = pd.DataFrame(history)

        model.eval()
        return model, history_df

    for epoch in range(
        start_epoch,
        NN_MAX_EPOCHS + 1
    ):
        t_epoch = time.time()
        model.train()

        # Retain the reference batch-shuffle formula with the replicate seed.
        tb = {
            p: make_batches(
                TEMP_DATASETS[('train', p)],
                NN_BATCH_TRAJECTORIES_PER_PHASE,
                True,
                seed + epoch +
                (0 if p == 'heating' else 10000)
            )
            for p in PHASES
        }

        db = {
            p: make_batches(
                DURATION_DATASETS[('train', p)],
                NN_BATCH_DURATION_ANCHORS_PER_PHASE,
                True,
                seed + 20000 + epoch +
                (0 if p == 'heating' else 10000)
            )
            for p in PHASES
        }

        nsteps = max(
            *[len(tb[p]) for p in PHASES],
            *[len(db[p]) for p in PHASES]
        )

        cyc_t = {
            p: cycle(tb[p])
            for p in PHASES
        }
        cyc_d = {
            p: cycle(db[p])
            for p in PHASES
        }

        acc = []

        for _ in range(nsteps):
            optimizer.zero_grad(set_to_none=True)

            phase_losses = {}

            for phase in PHASES:
                lt = trajectory_batch_loss(
                    model,
                    next(cyc_t[phase]),
                    phase
                )

                ld = duration_batch_loss(
                    model,
                    next(cyc_d[phase]),
                    phase
                )

                phase_losses[phase] = {
                    'trajectory': lt['trajectory'],
                    'rate': lt['rate'],
                    'temperature': lt['temperature'],
                    'duration': ld,
                    'total':
                        lt['trajectory'] + ld,
                }

            total = 0.5 * (
                phase_losses['heating']['total'] +
                phase_losses['cooling']['total']
            )

            scaler.scale(total).backward()
            scaler.unscale_(optimizer)

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                5.0
            )

            scaler.step(optimizer)
            scaler.update()

            acc.append({
                'total':
                    float(total.detach().cpu()),
                'trajectory':
                    float(
                        (
                            0.5 * (
                                phase_losses['heating']['trajectory'] +
                                phase_losses['cooling']['trajectory']
                            )
                        ).detach().cpu()
                    ),
                'duration_q_weighted':
                    float(
                        (
                            0.5 * (
                                phase_losses['heating']['duration'] +
                                phase_losses['cooling']['duration']
                            )
                        ).detach().cpu()
                    ),
            })

        tr = {
            k: float(
                np.mean([x[k] for x in acc])
            )
            for k in acc[0]
        }

        rec = {
            'random_seed': seed,
            'epoch': epoch,
            **{
                f'train_{k}': v
                for k, v in tr.items()
            },
            'lr':
                float(optimizer.param_groups[0]['lr']),
            'epoch_seconds':
                float(time.time() - t_epoch),
            'validation_performed':
                False,
        }

        do_val = (
            epoch == 1 or
            epoch % NN_VALIDATION_EVERY == 0 or
            epoch == NN_MAX_EPOCHS
        )

        if do_val:
            tval = time.time()

            vt = {
                p: evaluate_trajectory(
                    model,
                    TEMP_DATASETS[('valid', p)],
                    p
                )
                for p in PHASES
            }

            vd = {}
            pred = {}

            for p in PHASES:
                vd[p], pred[p] = duration_eval_metrics(
                    model,
                    DURATION_DATASETS[('valid', p)],
                    p
                )

            # Reference validation/model-selection criterion.
            val_duration_loss = 0.5 * (
                vd['heating']['duration_norm_mse_unweighted'] +
                vd['cooling']['duration_norm_mse_unweighted']
            )

            val_duration_mae = 0.5 * (
                vd['heating']['duration_MAE_min_unweighted'] +
                vd['cooling']['duration_MAE_min_unweighted']
            )

            val_rel = 0.5 * (
                vd['heating']['duration_Median_RAE_pct_unweighted'] +
                vd['cooling']['duration_Median_RAE_pct_unweighted']
            )

            val_traj = 0.5 * (
                vt['heating']['trajectory'] +
                vt['cooling']['trajectory']
            )

            scheduler.step(
                val_duration_loss
            )

            rec.update({
                'validation_performed':
                    True,
                'validation_seconds':
                    float(time.time() - tval),
                'val_duration_norm_mse_unweighted_balanced':
                    val_duration_loss,
                'val_duration_MAE_min_unweighted_balanced':
                    val_duration_mae,
                'val_duration_Median_RAE_pct_unweighted_balanced':
                    val_rel,
                'val_trajectory_balanced':
                    val_traj,
                'val_heating_MAE_min':
                    vd['heating']['duration_MAE_min_unweighted'],
                'val_cooling_MAE_min':
                    vd['cooling']['duration_MAE_min_unweighted'],
                'val_heating_Median_RAE_pct':
                    vd['heating']['duration_Median_RAE_pct_unweighted'],
                'val_cooling_Median_RAE_pct':
                    vd['cooling']['duration_Median_RAE_pct_unweighted'],
                'lr_after_scheduler':
                    float(optimizer.param_groups[0]['lr']),
            })

            improved = (
                val_duration_loss <
                best_val - NN_MIN_DELTA
            )

            if improved:
                best_val = val_duration_loss
                best_epoch = epoch
                stale_validations = 0

                save_best_seed_model(
                    paths['best'],
                    model,
                    seed,
                    epoch,
                    val_duration_loss,
                    rec
                )
            else:
                stale_validations += 1

            print(
                f'seed={seed} epoch={epoch:03d} '
                f'valDurLoss={val_duration_loss:.5f} '
                f'traj={val_traj:.4f} '
                f'MAE H/C='
                f'{vd["heating"]["duration_MAE_min_unweighted"]:.2f}/'
                f'{vd["cooling"]["duration_MAE_min_unweighted"]:.2f} min '
                f'| MedRAE H/C='
                f'{vd["heating"]["duration_Median_RAE_pct_unweighted"]:.1f}/'
                f'{vd["cooling"]["duration_Median_RAE_pct_unweighted"]:.1f}% '
                f'| lr={optimizer.param_groups[0]["lr"]:.2e} '
                f'| best={best_epoch}'
            )

        history.append(rec)

        pd.DataFrame(history).to_csv(
            paths['history'],
            index=False
        )

        torch.save({
            'random_seed':
                seed,
            'epoch':
                epoch,
            'model_state_dict':
                model.state_dict(),
            'optimizer_state_dict':
                optimizer.state_dict(),
            'scheduler_state_dict':
                scheduler.state_dict(),
            'scaler_state_dict':
                scaler.state_dict() if USE_AMP else None,
            'best_val':
                best_val,
            'best_epoch':
                best_epoch,
            'stale_validations':
                stale_validations,
            'history':
                history,
            'rng_state':
                capture_rng_state(),
            'shared_split_sha256':
                SHARED_SPLIT_SHA256,
            'batch_trajectories_per_phase':
                NN_BATCH_TRAJECTORIES_PER_PHASE,
            'batch_duration_anchors_per_phase':
                NN_BATCH_DURATION_ANCHORS_PER_PHASE,
        }, paths['resume'])

        if (
            do_val and
            stale_validations >=
            NN_EARLY_STOP_PATIENCE_VALIDATIONS
        ):
            print(
                f'Seed {seed}: early stopping after '
                f'{stale_validations} stale validation checkpoints.'
            )
            break

    if not paths['best'].exists():
        raise RuntimeError(
            f'Seed {seed}: training finished without a best-validation checkpoint.'
        )

    with open(
        paths['complete'],
        'w',
        encoding='utf-8'
    ) as f:
        json.dump({
            'random_seed':
                seed,
            'best_epoch':
                best_epoch,
            'best_validation_duration_norm_mse':
                best_val,
            'selection_metric':
                'balanced unweighted normalized validation duration MSE',
            'test_used_for_model_selection':
                False,
            'shared_split_sha256':
                SHARED_SPLIT_SHA256,
            'batch_trajectories_per_phase':
                NN_BATCH_TRAJECTORIES_PER_PHASE,
            'batch_duration_anchors_per_phase':
                NN_BATCH_DURATION_ANCHORS_PER_PHASE,
        }, f, indent=2)

    best = safe_torch_load(
        paths['best'],
        map_location=DEVICE
    )

    model.load_state_dict(
        best['state_dict']
    )
    model.eval()

    return model, pd.DataFrame(history)

# Load seed 42 from the original checkpoint.
CANONICAL_MODEL = UnifiedDirectedRateNN(
    int(CANONICAL_CKPT['n_inputs']),
    CANONICAL_CKPT['hidden_dims'],
    CANONICAL_CKPT['dropout']
).to(DEVICE)
CANONICAL_MODEL.load_state_dict(
    CANONICAL_CKPT['state_dict']
)
CANONICAL_MODEL.eval()

MODELS = {
    CANONICAL_SEED: CANONICAL_MODEL
}

HISTORIES = {}

canonical_history_path = (
    CANONICAL_TRAIN_RUN /
    'tables' /
    'unified_nn_training_history_score_weighted.csv'
)
if canonical_history_path.exists():
    HISTORIES[CANONICAL_SEED] = pd.read_csv(
        canonical_history_path
    )

# Train or resume the fthe pre-specified new seeds.
for seed in NEW_SEEDS:
    print('\n' + '=' * 80)
    print('TRAIN / LOAD SEED:', seed)
    print('=' * 80)

    model, history = train_seed_model(seed)
    MODELS[seed] = model
    HISTORIES[seed] = history

if set(MODELS) != set(SEEDS):
    raise RuntimeError(
        f'All five pre-specified seeds must be present before evaluation. '
        f'Have {sorted(MODELS)}, expected {SEEDS}.'
    )

print('\nAll pre-specified seed models are available:', sorted(MODELS))


## 7. Verify reference inference

Evaluate each seed with Notebook 03's completion-time procedure. First require the original checkpoint to reproduce its saved duration predictions. This separates inference consistency checks from variation introduced by new training runs.


In [ ]:


# Exact Notebook-03 evaluation completion-time procedure + batched NN TEST inference

def crossed(phase, T, threshold):
    return (
        float(T) >= float(threshold)
        if phase == 'heating'
        else float(T) <= float(threshold)
    )

def direction_valid(phase, dT):
    return (
        np.isfinite(dT) and
        (
            (dT > 0)
            if phase == 'heating'
            else (dT < 0)
        )
    )

def linear_crossing(
    h0, T0, h1, T1, threshold
):
    if abs(T1 - T0) < 1e-12:
        return float(h1)

    a = np.clip(
        (threshold - T0) / (T1 - T0),
        0, 1
    )

    return float(
        h0 + a * (h1 - h0)
    )

def estimate_crossing_from_source(
    phase,
    current_T,
    threshold,
    horizons,
    pred_T,
    current_delta_prev=np.nan
):
    h = np.asarray(horizons, float)
    y = np.asarray(pred_T, float)

    m = np.isfinite(h) & np.isfinite(y)
    h = h[m]
    y = y[m]

    if len(h) == 0:
        return np.nan, 'no_valid_trajectory', np.nan

    o = np.argsort(h)
    h = h[o]
    y = y[o]

    h0 = np.r_[0.0, h]
    y0 = np.r_[float(current_T), y]

    if (
        crossed(phase, current_T, threshold) and
        direction_valid(phase, current_delta_prev)
    ):
        return (
            0.0,
            'label_matched_at_source',
            float(h[-1])
        )

    ph, pt = h0[0], y0[0]

    for hh, tt in zip(h0[1:], y0[1:]):
        d = tt - pt

        if (
            crossed(phase, tt, threshold) and
            direction_valid(phase, d)
        ):
            if not crossed(phase, pt, threshold):
                return (
                    linear_crossing(
                        ph, pt, hh, tt, threshold
                    ),
                    'label_matched_interpolated_crossing',
                    float(h[-1])
                )

            return (
                float(hh),
                'label_matched_direction_at_horizon',
                float(h[-1])
            )

        ph, pt = hh, tt

    # Exact Notebook-03 final-trend extrapolation.
    n = min(
        EXTRAPOLATE_FINAL_N_POINTS,
        len(h0)
    )

    if n >= 3:
        slope, intercept = np.polyfit(
            h0[-n:],
            y0[-n:],
            1
        )

        good = (
            (
                phase == 'heating' and
                slope >
                EXTRAPOLATE_MIN_ABS_SLOPE_C_PER_MIN
            ) or
            (
                phase == 'cooling' and
                slope <
                -EXTRAPOLATE_MIN_ABS_SLOPE_C_PER_MIN
            )
        )

        if good:
            hp = (
                float(threshold) - intercept
            ) / slope

            cap = float(
                h0[-1] +
                EXTRAPOLATE_MAX_EXTRA_MIN_BY_PHASE[phase]
            )

            if (
                np.isfinite(hp) and
                hp > h0[-1]
            ):
                return (
                    float(min(hp, cap)),
                    (
                        'label_matched_extrapolated_final_trend'
                        if hp <= cap
                        else 'projected_beyond_cap_used_cap'
                    ),
                    float(h[-1])
                )

    return (
        float(h0[-1]),
        'no_label_match_used_last_horizon',
        float(h[-1])
    )

def source_to_anchor_remaining(
    source_crossing_min,
    asof_gap_min
):
    if not np.isfinite(source_crossing_min):
        return np.nan

    return float(
        max(
            0.0,
            float(source_crossing_min) -
            float(asof_gap_min)
        )
    )

def initial_rate_array(df):
    if 'Room_dTdt_now_bwd2_C_per_min' in df.columns:
        v = pd.to_numeric(
            df['Room_dTdt_now_bwd2_C_per_min'],
            errors='coerce'
        ).to_numpy(float)
    else:
        v = np.full(len(df), np.nan)

    if 'Room_dTdt_central_-5min_C_per_min' in df.columns:
        fb = pd.to_numeric(
            df['Room_dTdt_central_-5min_C_per_min'],
            errors='coerce'
        ).to_numpy(float)
    else:
        fb = np.zeros(len(df), dtype=float)

    bad = ~np.isfinite(v)
    v[bad] = fb[bad]
    v[~np.isfinite(v)] = 0.0

    return v

def integrate_numpy_batch(
    h,
    pred_directed,
    T0,
    sign,
    v0_phys
):
    pred_directed = np.asarray(
        pred_directed,
        float
    )
    T0 = np.asarray(T0, float)
    sign = np.asarray(sign, float)
    v0_phys = np.asarray(v0_phys, float)

    pred_phys = sign[:, None] * pred_directed
    dt = np.r_[
        h[0],
        np.diff(h)
    ].astype(float)

    prev = np.concatenate(
        [
            v0_phys[:, None],
            pred_phys[:, :-1]
        ],
        axis=1
    )

    return (
        T0[:, None] +
        np.cumsum(
            0.5 * dt[None, :] *
            (prev + pred_phys),
            axis=1
        )
    )

def make_expanded_anchor_frame(
    chunk,
    h
):
    H = len(h)

    expanded = (
        chunk.loc[
            chunk.index.repeat(H)
        ]
        .copy()
        .reset_index(drop=True)
    )

    expanded['horizon_min'] = np.tile(
        h,
        len(chunk)
    )

    expanded['target_timestamp'] = (
        pd.to_datetime(
            expanded['feature_source_timestamp']
        ) +
        pd.to_timedelta(
            expanded['horizon_min'],
            unit='m'
        )
    )

    return expanded

def predict_seed_hard(
    seed,
    model,
    anchors,
    chunk_size=1024
):
    records = []
    model.eval()

    for phase in PHASES:
        ap = (
            anchors[
                anchors['phase'].eq(phase)
            ]
            .copy()
            .reset_index(drop=True)
        )

        h = np.arange(
            STEP_MIN,
            INDIRECT_MAX_HORIZON_BY_PHASE[phase] + 1e-9,
            STEP_MIN,
            dtype=float
        )
        H = len(h)

        for lo in range(
            0,
            len(ap),
            chunk_size
        ):
            chunk = (
                ap.iloc[
                    lo:lo + chunk_size
                ]
                .copy()
                .reset_index(drop=True)
            )

            B = len(chunk)

            expanded = make_expanded_anchor_frame(
                chunk,
                h
            )

            X = transform_X(
                expanded
            )

            xt = torch.from_numpy(
                X
            ).to(
                DEVICE,
                dtype=torch.float32,
                non_blocking=True
            )

            with torch.inference_mode():
                pred_directed = (
                    model(xt)
                    .float()
                    .reshape(B, H)
                    .cpu()
                    .numpy()
                )

            del xt

            sign_arr = pd.to_numeric(
                chunk['thermal_mode_sign'],
                errors='coerce'
            ).to_numpy(float)

            T0_arr = pd.to_numeric(
                chunk['Room_T'],
                errors='coerce'
            ).to_numpy(float)

            v0_arr = initial_rate_array(
                chunk
            )

            pT_all = integrate_numpy_batch(
                h,
                pred_directed,
                T0_arr,
                sign_arr,
                v0_arr
            )

            for bi in range(B):
                row = chunk.iloc[bi]

                threshold = float(
                    row['Duration_Crossing_Threshold_C']
                )

                current_delta = (
                    float(v0_arr[bi]) *
                    STEP_MIN
                )

                cross, ctype, maxh = (
                    estimate_crossing_from_source(
                        phase,
                        float(T0_arr[bi]),
                        threshold,
                        h,
                        pT_all[bi],
                        current_delta
                    )
                )

                pred_anchor = (
                    source_to_anchor_remaining(
                        cross,
                        float(
                            row['asof_gap_min']
                        )
                    )
                )

                records.append({
                    'random_seed':
                        int(seed),
                    'anchor_key':
                        row['anchor_key'],
                    'phase':
                        phase,
                    'room':
                        row['room'],
                    'event_key':
                        row['event_key'],
                    'split':
                        row['split'],
                    'anchor_kind':
                        row.get(
                            'anchor_kind',
                            'anchor'
                        ),
                    'anchor_timestamp':
                        row['anchor_timestamp'],
                    'feature_source_timestamp':
                        row['feature_source_timestamp'],
                    'asof_gap_min':
                        row['asof_gap_min'],
                    'true_remaining_duration_min':
                        row['true_remaining_duration_min'],
                    'remaining_trajectory_quality_score':
                        row['remaining_trajectory_quality_score'],
                    'pred_remaining_duration_min':
                        pred_anchor,
                    'crossing_from_feature_source_min':
                        cross,
                    'crossing_type':
                        ctype,
                    'max_horizon_min':
                        maxh,
                })

    return pd.DataFrame(records)

EVAL_ANCHORS = (
    ANCHORS[
        ANCHORS['split'].eq('test')
    ]
    .copy()
    .reset_index(drop=True)
)

seed_pred_frames = []

for seed in SEEDS:
    print('Hard TEST inference — seed', seed)

    pred = predict_seed_hard(
        seed,
        MODELS[seed],
        EVAL_ANCHORS
    )

    if pred['anchor_key'].nunique() != EVAL_ANCHORS['anchor_key'].nunique():
        raise RuntimeError(
            f'Seed {seed}: incomplete TEST anchor coverage.'
        )

    if pred['pred_remaining_duration_min'].isna().any():
        raise RuntimeError(
            f'Seed {seed}: non-finite hard TEST prediction.'
        )

    seed_pred_frames.append(pred)

SEED_PRED = pd.concat(
    seed_pred_frames,
    ignore_index=True
)

# Seed-42 reference replay against the exact Notebook-03 file.
# Write the final prediction file after the reference reproduction check passes.

NB03_ALL = pd.read_csv(
    NB03_ALL_PRED_PATH
)

nb03_nn = (
    NB03_ALL[
        (NB03_ALL['split'].eq('test')) &
        (NB03_ALL['method'].eq('Unified_Derivative_NN'))
    ][
        [
            'anchor_key',
            'pred_remaining_duration_min'
        ]
    ]
    .drop_duplicates('anchor_key')
    .rename(columns={
        'pred_remaining_duration_min':
            'pred_nb03'
    })
)

seed42 = (
    SEED_PRED[
        SEED_PRED['random_seed'].eq(
            CANONICAL_SEED
        )
    ][
        [
            'anchor_key',
            'pred_remaining_duration_min'
        ]
    ]
    .rename(columns={
        'pred_remaining_duration_min':
            'pred_nb07_seed42'
    })
)

replay = seed42.merge(
    nb03_nn,
    on='anchor_key',
    how='outer',
    validate='one_to_one'
)

if replay.isna().any().any():
    raise RuntimeError(
        'Seed-42 replay anchor/support mismatch with Notebook 03.'
    )

replay['abs_diff_min'] = np.abs(
    replay['pred_nb07_seed42'] -
    replay['pred_nb03']
)

max_replay_diff = float(
    replay['abs_diff_min'].max()
)

print(
    'Seed-42 canonical replay max abs difference [min]:',
    max_replay_diff
)

if (
    STRICT_CANONICAL_REPLAY and
    max_replay_diff > CANONICAL_REPLAY_TOL_MIN
):
    raise RuntimeError(
        'Seed-42 replay failed. Multi-seed comparison would not be '
        'strictly anchored to the manuscript model.'
    )

print('Seed-42 Notebook-03 hard-prediction replay: PASSED')

# After verifying reference inference, retain the saved seed-42 predictions in summary
# tables to avoid differences from inference rounding.
_nb03_seed42_map = nb03_nn.set_index('anchor_key')['pred_nb03']
_seed42_mask = SEED_PRED['random_seed'].eq(CANONICAL_SEED)
SEED_PRED.loc[
    _seed42_mask,
    'pred_remaining_duration_min'
] = (
    SEED_PRED.loc[_seed42_mask, 'anchor_key']
    .map(_nb03_seed42_map)
    .to_numpy(float)
)

if SEED_PRED.loc[_seed42_mask, 'pred_remaining_duration_min'].isna().any():
    raise RuntimeError(
        'Failed to restore exact saved Notebook-03 seed-42 predictions.'
    )

# Save only after exact reference substitution.
SEED_PRED.to_csv(
    PRED_DIR / 'NB07_all_5seed_hard_duration_predictions.csv',
    index=False
)

print('Final seed-42 rows anchored to exact saved Notebook-03 predictions.')


## 8. Compare duration accuracy

Report every seed on the common test prediction keys used by the three main models. Export full-test neural results separately so that prediction coverage and common-case accuracy remain distinguishable.


In [ ]:


# Exact-common support and duration metrics for all five seeds

PAPER_METHODS = [
    'Direct_RoomPhase_ET',
    'Indirect_RoomPhase_ET',
    'Unified_Derivative_NN',
]

nb03_test = NB03_ALL[
    NB03_ALL['split'].eq('test')
].copy()

wide_paper = nb03_test.pivot_table(
    index='anchor_key',
    columns='method',
    values='pred_remaining_duration_min',
    aggfunc='first'
)

for m in PAPER_METHODS:
    if m not in wide_paper.columns:
        wide_paper[m] = np.nan

EXACT_COMMON_KEYS = set(
    wide_paper.dropna(
        subset=PAPER_METHODS
    ).index.astype(str)
)

EXPECTED_COMMON = private_value('nb07.expected_common')
EXPECTED_FULL = private_value('nb07.expected_full')

for phase in PHASES:
    n_common = int(
        EVAL_ANCHORS[
            EVAL_ANCHORS['phase'].eq(phase) &
            EVAL_ANCHORS['anchor_key'].astype(str).isin(
                EXACT_COMMON_KEYS
            )
        ]['anchor_key'].nunique()
    )
    n_full = int(
        EVAL_ANCHORS[
            EVAL_ANCHORS['phase'].eq(phase)
        ]['anchor_key'].nunique()
    )

    if n_common != EXPECTED_COMMON[phase]:
        raise RuntimeError(
            f'{phase}: exact-common anchor count changed: '
            f'{n_common} vs {EXPECTED_COMMON[phase]}'
        )
    if n_full != EXPECTED_FULL[phase]:
        raise RuntimeError(
            f'{phase}: full TEST anchor count changed: '
            f'{n_full} vs {EXPECTED_FULL[phase]}'
        )

def duration_metrics(g):
    true = pd.to_numeric(
        g['true_remaining_duration_min'],
        errors='coerce'
    ).to_numpy(float)

    pred = pd.to_numeric(
        g['pred_remaining_duration_min'],
        errors='coerce'
    ).to_numpy(float)

    good = np.isfinite(true) & np.isfinite(pred)
    true = true[good]
    pred = pred[good]

    if len(true) == 0:
        return pd.Series({
            'n': 0,
            'MAE_min': np.nan,
            'RMSE_min': np.nan,
            'Bias_min': np.nan,
            'MedianAE_min': np.nan,
            'P90AE_min': np.nan,
            'MeanRAE_pct': np.nan,
            'MedianRAE_pct': np.nan,
            'P90RAE_pct': np.nan,
            'RelativeBias_pct': np.nan,
        })

    e = pred - true
    ae = np.abs(e)
    denom = np.maximum(true, STEP_MIN)
    rae = 100.0 * ae / denom
    rse = 100.0 * e / denom

    return pd.Series({
        'n':
            len(e),
        'MAE_min':
            float(np.mean(ae)),
        'RMSE_min':
            float(np.sqrt(np.mean(e ** 2))),
        'Bias_min':
            float(np.mean(e)),
        'MedianAE_min':
            float(np.median(ae)),
        'P90AE_min':
            float(np.percentile(ae, 90)),
        'MeanRAE_pct':
            float(np.mean(rae)),
        'MedianRAE_pct':
            float(np.median(rae)),
        'P90RAE_pct':
            float(np.percentile(rae, 90)),
        'RelativeBias_pct':
            float(np.mean(rse)),
    })

COMMON_PRED = SEED_PRED[
    SEED_PRED['anchor_key']
    .astype(str)
    .isin(EXACT_COMMON_KEYS)
].copy()

SEED_METRICS_COMMON = (
    COMMON_PRED.groupby(
        ['phase', 'random_seed'],
        as_index=False
    )
    .apply(
        duration_metrics,
        include_groups=False
    )
    .reset_index(drop=True)
)

SEED_METRICS_FULL = (
    SEED_PRED.groupby(
        ['phase', 'random_seed'],
        as_index=False
    )
    .apply(
        duration_metrics,
        include_groups=False
    )
    .reset_index(drop=True)
)

SEED_METRICS_COMMON.to_csv(
    TABLE_DIR / 'PAPER_NB07_5seed_duration_metrics_EXACT_COMMON_SUPPORT.csv',
    index=False
)

SEED_METRICS_FULL.to_csv(
    TABLE_DIR / 'PAPER_NB07_5seed_duration_metrics_FULL_TEST_SUPPORT.csv',
    index=False
)

# Consistency check: reference seed-42 metrics must equal Notebook 03.

NB03_MAIN = pd.read_csv(
    NB03_MAIN_PATH
)

nb03_nn_main = NB03_MAIN[
    NB03_MAIN['method'].eq(
        'Unified_Derivative_NN'
    )
].copy()

seed42_main = SEED_METRICS_COMMON[
    SEED_METRICS_COMMON['random_seed'].eq(
        CANONICAL_SEED
    )
].copy()

metric_cols = [
    'n',
    'MAE_min',
    'RMSE_min',
    'Bias_min',
    'MedianAE_min',
    'P90AE_min',
    'MeanRAE_pct',
    'MedianRAE_pct',
    'P90RAE_pct',
    'RelativeBias_pct',
]

metric_check = seed42_main.merge(
    nb03_nn_main[
        ['phase'] + metric_cols
    ],
    on='phase',
    suffixes=('_nb07', '_nb03'),
    validate='one_to_one'
)

for c in metric_cols:
    a = pd.to_numeric(
        metric_check[f'{c}_nb07'],
        errors='coerce'
    ).to_numpy(float)

    b = pd.to_numeric(
        metric_check[f'{c}_nb03'],
        errors='coerce'
    ).to_numpy(float)

    if not np.allclose(
        a, b,
        rtol=0.0,
        atol=1e-8,
        equal_nan=True
    ):
        raise RuntimeError(
            f'Seed-42 metric reproduction failed for {c}.'
        )

print('Seed-42 exact-common Notebook-03 metric reproduction: PASSED')

print('\nFive-seed exact-common duration metrics:')
display(
    SEED_METRICS_COMMON.sort_values(
        ['phase', 'random_seed']
    ).round(3)
)


## 9. Compare event-averaged errors

Average errors within each event and then across events for each seed. This complements prediction-time averages, in which longer events contribute more repeated predictions. The code uses `MacroEvent` for event-averaged outputs.


In [ ]:


# Event-averaged metrics for each seed

event_rows = []

for (
    phase,
    seed,
    event_key
), g in COMMON_PRED.groupby(
    ['phase', 'random_seed', 'event_key'],
    sort=False
):
    true = pd.to_numeric(
        g['true_remaining_duration_min'],
        errors='coerce'
    ).to_numpy(float)

    pred = pd.to_numeric(
        g['pred_remaining_duration_min'],
        errors='coerce'
    ).to_numpy(float)

    e = pred - true
    ae = np.abs(e)
    denom = np.maximum(true, STEP_MIN)
    rae = 100.0 * ae / denom

    event_rows.append({
        'phase':
            phase,
        'random_seed':
            int(seed),
        'event_key':
            event_key,
        'n_anchors':
            len(g),
        'Event_MAE_min':
            float(np.mean(ae)),
        'Event_RMSE_min':
            float(np.sqrt(np.mean(e ** 2))),
        'Event_MeanRAE_pct':
            float(np.mean(rae)),
    })

PER_EVENT = pd.DataFrame(
    event_rows
)

MACRO_BY_SEED = (
    PER_EVENT.groupby(
        ['phase', 'random_seed'],
        as_index=False
    )
    .agg(
        n_events=(
            'event_key',
            'nunique'
        ),
        MacroEvent_MAE_min=(
            'Event_MAE_min',
            'mean'
        ),
        MacroEvent_RMSE_min=(
            'Event_RMSE_min',
            'mean'
        ),
        MacroEvent_MeanRAE_pct=(
            'Event_MeanRAE_pct',
            'mean'
        ),
    )
)

PER_EVENT.to_csv(
    TABLE_DIR / 'NB07_5seed_per_event_metrics_EXACT_COMMON_SUPPORT.csv',
    index=False
)

MACRO_BY_SEED.to_csv(
    TABLE_DIR / 'PAPER_NB07_5seed_event_macro_metrics_EXACT_COMMON_SUPPORT.csv',
    index=False
)

print('Five-seed event-macro metrics:')
display(
    MACRO_BY_SEED.sort_values(
        ['phase', 'random_seed']
    ).round(3)
)


## 10. Summarize aggregate training variation

Report the across-seed mean, sample standard deviation, range and coefficient of variation. Compare each seed with the same fixed Direct ET and Temperature ET results on common cases. Retain all seeds without selecting a test-performance winner.


In [ ]:


# Across-seed stability + fixed-baseline comparison

seed_summary_source = SEED_METRICS_COMMON.merge(
    MACRO_BY_SEED,
    on=['phase', 'random_seed'],
    how='left',
    validate='one_to_one'
)

STABILITY_METRICS = [
    'MAE_min',
    'RMSE_min',
    'MedianAE_min',
    'MedianRAE_pct',
    'MacroEvent_MAE_min',
]

stability_rows = []

for phase in PHASES:
    g = seed_summary_source[
        seed_summary_source['phase'].eq(phase)
    ].copy()

    if set(g['random_seed'].astype(int)) != set(SEEDS):
        raise RuntimeError(
            f'{phase}: all five pre-specified seeds must be reported.'
        )

    for metric in STABILITY_METRICS:
        x = pd.to_numeric(
            g[metric],
            errors='coerce'
        ).to_numpy(float)

        mean = float(np.mean(x))
        sd = float(np.std(x, ddof=1))

        stability_rows.append({
            'phase':
                phase,
            'metric':
                metric,
            'n_seeds':
                len(x),
            'seed_mean':
                mean,
            'seed_SD':
                sd,
            'seed_min':
                float(np.min(x)),
            'seed_max':
                float(np.max(x)),
            'seed_range':
                float(np.max(x) - np.min(x)),
            'seed_CV_pct':
                float(
                    100.0 * sd / abs(mean)
                ) if abs(mean) > 1e-12 else np.nan,
            'canonical_seed42_value':
                float(
                    g.loc[
                        g['random_seed'].eq(CANONICAL_SEED),
                        metric
                    ].iloc[0]
                ),
        })

STABILITY_SUMMARY = pd.DataFrame(
    stability_rows
)

STABILITY_SUMMARY.to_csv(
    TABLE_DIR / 'PAPER_NB07_across_seed_stability_summary.csv',
    index=False
)

# Fixed baseline MAEs from Notebook 03.
baseline_mae = (
    NB03_MAIN[
        NB03_MAIN['method'].isin(
            [
                'Direct_RoomPhase_ET',
                'Indirect_RoomPhase_ET'
            ]
        )
    ][
        ['phase', 'method', 'MAE_min']
    ]
    .copy()
)

baseline_rows = []

for phase in PHASES:
    seed_rows = SEED_METRICS_COMMON[
        SEED_METRICS_COMMON['phase'].eq(phase)
    ].copy()

    for baseline_method in [
        'Direct_RoomPhase_ET',
        'Indirect_RoomPhase_ET'
    ]:
        b = float(
            baseline_mae.loc[
                (baseline_mae['phase'].eq(phase)) &
                (baseline_mae['method'].eq(baseline_method)),
                'MAE_min'
            ].iloc[0]
        )

        seed_mae = pd.to_numeric(
            seed_rows['MAE_min'],
            errors='coerce'
        ).to_numpy(float)

        deltas = seed_mae - b

        baseline_rows.append({
            'phase':
                phase,
            'baseline_method':
                baseline_method,
            'baseline_MAE_min':
                b,
            'n_seeds':
                len(seed_mae),
            'seeds_with_lower_MAE_than_baseline':
                int(np.sum(seed_mae < b)),
            'fraction_seeds_with_lower_MAE':
                float(np.mean(seed_mae < b)),
            'best_seed_MAE_min':
                float(np.min(seed_mae)),
            'worst_seed_MAE_min':
                float(np.max(seed_mae)),
            'mean_seed_delta_vs_baseline_min':
                float(np.mean(deltas)),
            'worst_seed_delta_vs_baseline_min':
                float(np.max(deltas)),
            'best_seed_delta_vs_baseline_min':
                float(np.min(deltas)),
        })

BASELINE_STABILITY = pd.DataFrame(
    baseline_rows
)

BASELINE_STABILITY.to_csv(
    TABLE_DIR / 'PAPER_NB07_seed_stability_vs_frozen_baselines.csv',
    index=False
)

print('Across-seed stability:')
display(
    STABILITY_SUMMARY.round(4)
)

print('\nFive-seed MAE comparison with frozen paper baselines:')
display(
    BASELINE_STABILITY.round(3)
)


## 11. Examine prediction-level dispersion

For each common prediction key, calculate the standard deviation of its five duration estimates. This describes seed-to-seed disagreement at that prediction time; it is not a calibrated confidence or prediction interval.


In [ ]:


# Prediction dispersion across the five seeds

dispersion_wide = (
    COMMON_PRED.pivot(
        index=[
            'anchor_key',
            'phase',
            'room',
            'event_key',
            'true_remaining_duration_min'
        ],
        columns='random_seed',
        values='pred_remaining_duration_min'
    )
    .reset_index()
)

for seed in SEEDS:
    if seed not in dispersion_wide.columns:
        raise RuntimeError(
            f'Missing seed {seed} from prediction-dispersion table.'
        )

seed_matrix = dispersion_wide[
    SEEDS
].to_numpy(float)

if not np.isfinite(seed_matrix).all():
    raise RuntimeError(
        'Non-finite prediction in seed-dispersion table.'
    )

dispersion_wide['seed_prediction_mean_min'] = (
    np.mean(seed_matrix, axis=1)
)
dispersion_wide['seed_prediction_SD_min'] = (
    np.std(seed_matrix, axis=1, ddof=1)
)
dispersion_wide['seed_prediction_range_min'] = (
    np.max(seed_matrix, axis=1) -
    np.min(seed_matrix, axis=1)
)

DISPERSION_SUMMARY = (
    dispersion_wide.groupby(
        'phase',
        as_index=False
    )
    .agg(
        anchors=(
            'anchor_key',
            'nunique'
        ),
        mean_seed_SD_min=(
            'seed_prediction_SD_min',
            'mean'
        ),
        median_seed_SD_min=(
            'seed_prediction_SD_min',
            'median'
        ),
        P90_seed_SD_min=(
            'seed_prediction_SD_min',
            lambda x: float(
                np.percentile(x, 90)
            )
        ),
        max_seed_SD_min=(
            'seed_prediction_SD_min',
            'max'
        ),
        mean_seed_range_min=(
            'seed_prediction_range_min',
            'mean'
        ),
        P90_seed_range_min=(
            'seed_prediction_range_min',
            lambda x: float(
                np.percentile(x, 90)
            )
        ),
    )
)

dispersion_wide.to_csv(
    TABLE_DIR / 'NB07_anchor_level_5seed_prediction_dispersion.csv',
    index=False
)

DISPERSION_SUMMARY.to_csv(
    TABLE_DIR / 'PAPER_NB07_prediction_dispersion_summary.csv',
    index=False
)

print('Anchor-level random-seed prediction dispersion:')
display(
    DISPERSION_SUMMARY.round(3)
)


## 12. Compare temperature and rate accuracy

Evaluate every seed on the same reference test trajectory rows. Treat these metrics as secondary to duration accuracy and retain the original checkpoint-selection criterion.


In [ ]:


# Secondary TEST trajectory stability

def predict_test_trajectory_seed(
    seed,
    model
):
    rows = []
    model.eval()

    with torch.inference_mode():
        for phase in PHASES:
            ds = TEMP_DATASETS[
                ('test', phase)
            ]

            for batch in make_batches(
                ds,
                512,
                False
            ):
                z, pv, pT = trajectory_batch_forward(
                    model,
                    batch
                )

                pv = pv.cpu().numpy()
                pT = pT.cpu().numpy()
                valid = z['valid'].cpu().numpy()

                for b, item in enumerate(batch):
                    n = int(
                        valid[b].sum()
                    )

                    for rid, v, t in zip(
                        item['row_ids'][:n],
                        pv[b, :n],
                        pT[b, :n]
                    ):
                        rows.append({
                            'random_seed':
                                int(seed),
                            '_row_id':
                                int(rid),
                            'phase':
                                phase,
                            'pred_directed_rate':
                                float(v),
                            'pred_T':
                                float(t),
                        })

    return pd.DataFrame(rows)

trajectory_frames = []

for seed in SEEDS:
    print('Trajectory TEST inference — seed', seed)
    trajectory_frames.append(
        predict_test_trajectory_seed(
            seed,
            MODELS[seed]
        )
    )

TRAJ_PRED = pd.concat(
    trajectory_frames,
    ignore_index=True
)

truth_cols = [
    '_row_id',
    'phase',
    'room',
    'event_key',
    'split',
    'target_context',
    'horizon_min',
    'target_Room_T',
    'target_directed_rate_C_per_min',
]

TRAJ_PRED = TRAJ_PRED.merge(
    DATA[truth_cols],
    on=['_row_id', 'phase'],
    how='left',
    validate='many_to_one'
)

TRAJ_PRED['T_error_C'] = (
    pd.to_numeric(
        TRAJ_PRED['pred_T'],
        errors='coerce'
    ) -
    pd.to_numeric(
        TRAJ_PRED['target_Room_T'],
        errors='coerce'
    )
)

TRAJ_PRED['rate_error_C_per_min'] = (
    pd.to_numeric(
        TRAJ_PRED['pred_directed_rate'],
        errors='coerce'
    ) -
    pd.to_numeric(
        TRAJ_PRED['target_directed_rate_C_per_min'],
        errors='coerce'
    )
)

# Retain inside-event targets for the primary temperature comparison.
TRAJ_INSIDE = TRAJ_PRED[
    TRAJ_PRED['target_context'].astype(str).eq(
        'inside_event'
    )
].copy()

def trajectory_metrics(g):
    te = pd.to_numeric(
        g['T_error_C'],
        errors='coerce'
    ).dropna().to_numpy(float)

    re = pd.to_numeric(
        g['rate_error_C_per_min'],
        errors='coerce'
    ).dropna().to_numpy(float)

    return pd.Series({
        'n_temperature_rows':
            len(te),
        'T_MAE_C':
            float(np.mean(np.abs(te))),
        'T_RMSE_C':
            float(np.sqrt(np.mean(te ** 2))),
        'T_Bias_C':
            float(np.mean(te)),
        'n_rate_rows':
            len(re),
        'Rate_MAE_C_per_min':
            float(np.mean(np.abs(re))),
        'Rate_RMSE_C_per_min':
            float(np.sqrt(np.mean(re ** 2))),
        'Rate_Bias_C_per_min':
            float(np.mean(re)),
    })

TRAJ_METRICS = (
    TRAJ_INSIDE.groupby(
        ['phase', 'random_seed'],
        as_index=False
    )
    .apply(
        trajectory_metrics,
        include_groups=False
    )
    .reset_index(drop=True)
)

TRAJ_METRICS.to_csv(
    TABLE_DIR / 'PAPER_NB07_5seed_TEST_inside_event_trajectory_metrics.csv',
    index=False
)

print('Five-seed secondary inside-event trajectory metrics:')
display(
    TRAJ_METRICS.sort_values(
        ['phase', 'random_seed']
    ).round(4)
)


## 13. Plot training-stability summaries

Show all seed outcomes, fixed ET reference errors, prediction-level dispersion and available validation-loss histories. Figures retain every evaluated replicate.


In [ ]:


# Paper-ready stability figures

# Figure A — per-seed MAE with fixed baseline reference lines.
fig, axes = plt.subplots(
    1, 2,
    figsize=(12.5, 4.8),
    constrained_layout=True
)

for ax, phase in zip(axes, PHASES):
    d = (
        SEED_METRICS_COMMON[
            SEED_METRICS_COMMON['phase'].eq(phase)
        ]
        .sort_values('random_seed')
    )

    ax.plot(
        d['random_seed'],
        d['MAE_min'],
        marker='o',
        label='Unified NN seeds'
    )

    for method, label in [
        ('Direct_RoomPhase_ET', 'Direct ET'),
        ('Indirect_RoomPhase_ET', 'Indirect ET'),
    ]:
        b = float(
            NB03_MAIN.loc[
                (NB03_MAIN['phase'].eq(phase)) &
                (NB03_MAIN['method'].eq(method)),
                'MAE_min'
            ].iloc[0]
        )

        ax.axhline(
            b,
            linestyle='--',
            label=label
        )

    ax.set_xticks(SEEDS)
    ax.set_xlabel('Random seed')
    ax.set_ylabel('MAE (min)')
    ax.set_title(
        f'{phase.capitalize()} — exact-common TEST'
    )
    ax.grid(alpha=.25)
    ax.legend(frameon=False)

fig.savefig(
    FIG_DIR / 'PAPER_NB07_5seed_MAE_vs_frozen_baselines.png',
    dpi=300,
    bbox_inches='tight'
)
plt.show()

# Figure B — anchor-level seed prediction dispersion.
fig, axes = plt.subplots(
    1, 2,
    figsize=(11.5, 4.6),
    constrained_layout=True
)

for ax, phase in zip(axes, PHASES):
    x = pd.to_numeric(
        dispersion_wide.loc[
            dispersion_wide['phase'].eq(phase),
            'seed_prediction_SD_min'
        ],
        errors='coerce'
    ).dropna().to_numpy(float)

    ax.hist(
        x,
        bins=30
    )
    ax.set_xlabel(
        'Across-seed prediction SD (min)'
    )
    ax.set_ylabel('Anchors')
    ax.set_title(
        phase.capitalize()
    )
    ax.grid(axis='y', alpha=.25)

fig.savefig(
    FIG_DIR / 'PAPER_NB07_anchor_prediction_seed_dispersion.png',
    dpi=300,
    bbox_inches='tight'
)
plt.show()

# Figure C — validation-duration-loss trajectories.
fig, axes = plt.subplots(
    1, 2,
    figsize=(12.5, 4.8),
    constrained_layout=True
)

for ax_i, metric in enumerate([
    'val_duration_norm_mse_unweighted_balanced',
    'val_duration_MAE_min_unweighted_balanced'
]):
    ax = axes[ax_i]

    for seed in SEEDS:
        h = HISTORIES.get(seed)
        if h is None or len(h) == 0:
            continue

        h = h.copy()

        if 'validation_performed' in h.columns:
            _vp = h['validation_performed']
            if _vp.dtype == bool:
                _mask = _vp
            else:
                _mask = (
                    _vp.astype(str)
                    .str.strip()
                    .str.lower()
                    .eq('true')
                )
            h = h[_mask]

        if metric not in h.columns:
            continue

        ax.plot(
            h['epoch'],
            h[metric],
            marker='o',
            markersize=2,
            label=f'seed {seed}'
        )

    ax.set_xlabel('Epoch')
    ax.set_ylabel(
        'Dimensionless validation duration MSE'
        if 'norm_mse' in metric
        else 'Validation duration MAE (min)'
    )
    ax.grid(alpha=.25)
    ax.legend(frameon=False)

axes[0].set_title('VALID model-selection metric')
axes[1].set_title('VALID duration MAE')

fig.savefig(
    FIG_DIR / 'PAPER_NB07_multiseed_validation_histories.png',
    dpi=300,
    bbox_inches='tight'
)
plt.show()

print('Figures saved to:', FIG_DIR)


## 14. Save reproducibility records

Record the data and split fingerprints, original and replicate checkpoints, selected epochs, feature order, training settings, target scales, software/hardware environment and output-file fingerprints. Require all specified seeds before marking the analysis complete.


In [ ]:


# Reproducibility manifest and final integrity checks

checkpoint_rows = []

for seed in SEEDS:
    if seed == CANONICAL_SEED:
        p = CANONICAL_CHECKPOINT
        ck = CANONICAL_CKPT
        source = 'frozen PAPER-FINAL canonical checkpoint'
    else:
        p = seed_paths(seed)['best']
        if not p.exists():
            raise FileNotFoundError(
                f'Missing completed checkpoint for seed {seed}: {p}'
            )
        ck = safe_torch_load(
            p,
            map_location='cpu'
        )
        source = 'Notebook-07 seed replicate'

        if int(ck.get('random_seed', -1)) != seed:
            raise RuntimeError(
                f'Seed {seed}: checkpoint metadata seed mismatch.'
            )

        if ck.get('shared_split_sha256') != SHARED_SPLIT_SHA256:
            raise RuntimeError(
                f'Seed {seed}: checkpoint split mismatch.'
            )

        if int(ck.get(
            'batch_trajectories_per_phase',
            -1
        )) != NN_BATCH_TRAJECTORIES_PER_PHASE:
            raise RuntimeError(
                f'Seed {seed}: trajectory batch size changed.'
            )

        if int(ck.get(
            'batch_duration_anchors_per_phase',
            -1
        )) != NN_BATCH_DURATION_ANCHORS_PER_PHASE:
            raise RuntimeError(
                f'Seed {seed}: duration batch size changed.'
            )

    checkpoint_rows.append({
        'random_seed':
            int(seed),
        'source':
            source,
        'checkpoint':
            str(p),
        'checkpoint_sha256':
            sha256_file(p),
        'best_epoch':
            int(ck.get('best_epoch', -1)),
        'best_validation_duration_norm_mse':
            float(
                ck.get(
                    'best_validation_duration_norm_mse',
                    np.nan
                )
            ),
    })

CHECKPOINT_SUMMARY = pd.DataFrame(
    checkpoint_rows
)

if set(
    CHECKPOINT_SUMMARY['random_seed'].astype(int)
) != set(SEEDS):
    raise RuntimeError(
        'Final manifest does not contain all pre-specified seeds.'
    )

CHECKPOINT_SUMMARY.to_csv(
    TABLE_DIR / 'PAPER_NB07_seed_checkpoint_summary.csv',
    index=False
)

# Package/runtime environment.
environment = {
    'python':
        sys.version,
    'platform':
        platform.platform(),
    'numpy':
        np.__version__,
    'pandas':
        pd.__version__,
    'scikit_learn':
        sklearn.__version__,
    'torch':
        torch.__version__,
    'joblib':
        joblib.__version__,
    'device':
        str(DEVICE),
    'gpu_name':
        GPU_NAME,
    'gpu_memory_GB':
        GPU_MEMORY_GB,
    'torch_cuda_version':
        torch.version.cuda,
    'cudnn_version':
        (
            torch.backends.cudnn.version()
            if torch.backends.cudnn.is_available()
            else None
        ),
    'amp_enabled':
        USE_AMP,
    'strict_determinism':
        STRICT_DETERMINISM,
}

prediction_path = (
    PRED_DIR /
    'NB07_all_5seed_hard_duration_predictions.csv'
)

manifest = {
    'notebook':
        '07_multiseed_stability_and_reproducibility_PAPER_FINAL',
    'role':
        'random-seed stability analysis of frozen Strategy III; no model selection from TEST',
    'base_dir':
        str(BASE_DIR),
    'uses_balanced_final':
        False,
    'dataset_dir':
        str(DATA_DIR),
    'shared_split_sha256':
        SHARED_SPLIT_SHA256,
    'canonical_seed':
        CANONICAL_SEED,
    'pre_specified_seed_set':
        SEEDS,
    'seed42_retrained':
        False,
    'all_seeds_reported':
        True,
    'canonical_training_run':
        str(CANONICAL_TRAIN_RUN),
    'canonical_notebook03_run':
        str(NOTEBOOK03_RUN),
    'canonical_seed42_replay_max_abs_diff_min':
        max_replay_diff,
    'canonical_reference_hardware': {
        'gpu':
            CANONICAL_GPU_REFERENCE,
        'reported_memory_GB':
            14.6,
        'trajectory_batch_per_phase':
            256,
        'duration_batch_per_phase':
            256,
    },
    'current_runtime':
        environment,
    'model': {
        'scope':
            'one room-blind network across heating and cooling',
        'input_features':
            list(MODEL_INPUTS),
        'n_inputs':
            len(MODEL_INPUTS),
        'hidden_dims':
            NN_HIDDEN_DIMS,
        'activation':
            'tanh',
        'dropout':
            NN_DROPOUT,
        'output':
            'directed rate v = thermal_mode_sign * dT/dt',
        'duration_head':
            False,
        'slope_loss':
            False,
        'q_weighted_duration_loss':
            True,
        'inverse_frequency_horizon_weights':
            False,
        'inverse_frequency_remaining_weights':
            False,
    },
    'optimization': {
        'optimizer':
            'AdamW',
        'learning_rate':
            NN_LR,
        'weight_decay':
            NN_WEIGHT_DECAY,
        'trajectory_batch_per_phase':
            NN_BATCH_TRAJECTORIES_PER_PHASE,
        'duration_batch_per_phase':
            NN_BATCH_DURATION_ANCHORS_PER_PHASE,
        'phase_balance':
            {'heating': 0.5, 'cooling': 0.5},
        'max_epochs':
            NN_MAX_EPOCHS,
        'validation_every_epochs':
            NN_VALIDATION_EVERY,
        'selection_metric':
            'balanced unweighted normalized validation duration MSE',
        'scheduler':
            'ReduceLROnPlateau',
        'scheduler_patience_validations':
            NN_SCHEDULER_PATIENCE_VALIDATIONS,
        'scheduler_factor':
            NN_SCHEDULER_FACTOR,
        'minimum_lr':
            NN_MIN_LR,
        'early_stop_patience_validations':
            NN_EARLY_STOP_PATIENCE_VALIDATIONS,
        'minimum_improvement':
            NN_MIN_DELTA,
        'gradient_clip':
            5.0,
        'test_used_for_model_selection':
            False,
    },
    'loss_scales':
        LOSS_SCALES,
    'soft_crossing': {
        'temp_beta_C':
            SOFT_TEMP_BETA_C,
        'direction_beta_C_by_phase':
            SOFT_DIRECTION_BETA_BY_PHASE,
    },
    'hard_converter': {
        'final_n_points':
            EXTRAPOLATE_FINAL_N_POINTS,
        'minimum_abs_slope_C_per_min':
            EXTRAPOLATE_MIN_ABS_SLOPE_C_PER_MIN,
        'max_extra_min_by_phase':
            EXTRAPOLATE_MAX_EXTRA_MIN_BY_PHASE,
    },
    'checkpoints':
        CHECKPOINT_SUMMARY.to_dict(
            orient='records'
        ),
    'prediction_file':
        str(prediction_path),
    'prediction_file_sha256':
        sha256_file(prediction_path),
    'primary_support':
        'Notebook-03 exact-common TEST anchors',
    'secondary_support':
        'full unified-NN TEST support',
    'across_seed_interpretation':
        (
            'Across-seed SD/range quantify stochastic training stability only; '
            'they are not calibrated predictive confidence intervals.'
        ),
}

manifest_path = (
    OUT_ROOT /
    'NB07_multiseed_reproducibility_manifest.json'
)

with open(
    manifest_path,
    'w',
    encoding='utf-8'
) as f:
    json.dump(
        manifest,
        f,
        indent=2,
        default=str
    )

with open(
    OUT_ROOT /
    'NB07_runtime_environment.json',
    'w',
    encoding='utf-8'
) as f:
    json.dump(
        environment,
        f,
        indent=2,
        default=str
    )

pd.DataFrame({
    'feature_order':
        np.arange(len(MODEL_INPUTS)),
    'feature':
        MODEL_INPUTS,
}).to_csv(
    OUT_ROOT /
    'NB07_exact_model_feature_order.csv',
    index=False
)

status = pd.DataFrame([
    {
        'check':
            'Uses PAPER-TEMP rather than PAPER-TEMP-NEW',
        'status':
            'PASS',
    },
    {
        'check':
            'Shared split matches Notebook 01/02/03',
        'status':
            'PASS',
    },
    {
        'check':
            'Feature order matches frozen seed-42 preprocessor',
        'status':
            'PASS',
    },
    {
        'check':
            'TRAIN-only preprocessing reconstructed exactly',
        'status':
            'PASS',
    },
    {
        'check':
            'Loss scales and soft-crossing parameters match seed 42',
        'status':
            'PASS',
    },
    {
        'check':
            'Seed 42 loaded from frozen checkpoint, not retrained',
        'status':
            'PASS',
    },
    {
        'check':
            'All five pre-specified seeds reported',
        'status':
            'PASS',
    },
    {
        'check':
            'Canonical 256/256 batch sizes preserved',
        'status':
            'PASS',
    },
    {
        'check':
            'No BALANCED_FINAL inverse-frequency weights',
        'status':
            'PASS',
    },
    {
        'check':
            'Notebook-03 hard converter preserved',
        'status':
            'PASS',
    },
    {
        'check':
            'Seed-42 hard predictions reproduce Notebook 03',
        'status':
            'PASS',
    },
    {
        'check':
            'TEST never used for checkpoint/seed selection',
        'status':
            'PASS',
    },
])

status.to_csv(
    AUDIT_DIR /
    'NB07_final_integrity_checks.csv',
    index=False
)

print('\nFINAL NOTEBOOK-07 INTEGRITY STATUS')
display(status)

print('\nSeed checkpoints:')
display(
    CHECKPOINT_SUMMARY
)

print('\nManifest:', manifest_path)
print('Output root:', OUT_ROOT)


## Interpretation

Distinguish stability of aggregate model errors from agreement on individual prediction times. Report all replicate outcomes, including outliers, and retain seed 42 as the reference model. A lower error across the tested seeds supports a comparison under this training procedure and test cohort; it does not establish universal superiority or cross-building transfer.

Use Notebook 04's event-bootstrap analysis for sampling variation and this notebook for training variation. Neither supplies calibrated uncertainty for an individual future completion time. The component variants in Notebook 05 are not each repeated across this seed set.
